In [ ]:
import glob
import io
import itertools
import math
import os
import pickle
import subprocess
import warnings
from pathlib import Path
from statistics import median

import boto3
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from scipy.stats import norm
from supervenn import supervenn
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

%config InlineBackend.figure_format = 'png'
plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 200

sns.set_style("ticks")
sns.axes_style("white")

# Cohort assembly originally ran against an internal cohort database, configured
# here with an API key. The public pipeline reads prepared tables directly, so no
# client and no credential is needed.

import pyreadr
import rpy2.robjects as ro
from biomart import BiomartServer
from neuroCombat import neuroCombat
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.vectors import ListVector, DataFrame, Matrix, FloatVector, IntVector, StrVector, BoolVector
from scipy.interpolate import interp1d
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# calc_cells() lived here. It wrapped an internal deconvolution library to turn
# expression into cell-type percentages. That library is not public and the helper
# was already unused (its import had been commented out), so it is removed.

# Data roots. These lived on internal shared storage; point them at wherever the
# prepared tables and reference files are staged.
FL_DATA_ROOT = os.environ.get("FL_DATA_ROOT", ".")
FL_SHARED_ROOT = os.environ.get("FL_SHARED_ROOT", ".")

In [ ]:
# ── Public compatibility shim ─────────────────────────────────────────────────
# This notebook was written against an internal BostonGene analysis library and an
# internal cohort database, neither of which is public. Every name it borrowed from
# them is redefined below on top of public packages, so the notebook remains a
# readable record of the analysis.
#
# Two of these are NOT numerically identical to the originals and are marked as such.
# Do not treat their output as reproducing the published values.

from sklearn.manifold import TSNE
from umap import UMAP


def read_dataset(path):
    """Read a tab-separated expression/annotation table (internal utils.read_dataset)."""
    return pd.read_csv(path, sep="\t")


def read_gene_sets(path):
    """Parse a GMT file into {set_name: [genes]} (internal gsea.read_gene_sets)."""
    gene_sets = {}
    with open(path) as fh:
        for line in fh:
            fields = line.rstrip("\n").split("\t")
            if len(fields) > 2:
                gene_sets[fields[0]] = [g for g in fields[2:] if g]
    return gene_sets


def median_scale(data, axis=0, clip=None):
    """Median-centre and robustly scale (internal utils.median_scale).

    axis=0 scales over columns, axis=1 over rows -- the internal convention, kept.

    NOT VERIFIED against the original: the internal function's scale factor is not
    published. This uses the median absolute deviation. Centring is exact; the
    spread of the returned values may differ from the published figures.
    """
    axis_arg = 0 if axis == 0 else 1
    centred = data.sub(data.median(axis=axis_arg), axis=1 - axis_arg)
    mad = centred.abs().median(axis=axis_arg).replace(0, np.nan)
    scaled = centred.div(mad, axis=1 - axis_arg)
    return scaled.clip(-clip, clip) if clip else scaled


def ssgsea_formula(data, gene_sets, rank_method="max"):
    """ssGSEA scores (internal ssgsea.ssgsea_formula).

    NOT NUMERICALLY IDENTICAL: this delegates to gseapy, whose normalisation differs
    from the internal implementation the published scores were computed with.
    """
    import gseapy
    res = gseapy.ssgsea(data=data, gene_sets=gene_sets, outdir=None,
                        no_plot=True, processes=4)
    return res.res2d.pivot(index="Term", columns="Name", values="NES").astype(float)


def run_progeny(data, organism="Human"):
    """PROGENy pathway scores (internal pathway_scoring.run_progeny)."""
    raise NotImplementedError(
        "PROGENy scoring used an internal wrapper around the PROGENy model weights. "
        "Substitute decoupler (decoupler.run_mlm with decoupler.get_progeny) or the "
        "PROGENy Bioconductor package; results will not match the published scores "
        "exactly, so re-derive any downstream figure that depends on them."
    )


# The three plot helpers below are lifted verbatim from
# harmonization_benchmark.ipynb cell 4, where they already replace the
# internal plotting utilities.
# ── Custom dimensionality reduction plot helpers ──────────────────────────────
import math as _math

def _lookup_color(palette: dict, g) -> str:
    """Dict lookup that handles float(nan) and None keys."""
    if g in palette:
        return palette[g]
    if isinstance(g, float) and _math.isnan(g):
        for k in palette:
            if isinstance(k, float) and _math.isnan(k):
                return palette[k]
    return palette.get(str(g), "#888888")


def _draw_grouped_legend(ax, unique_groups, colors: dict, palette_groups: dict) -> None:
    """Render legend with bold group subtitles and visual separators."""
    from matplotlib.lines import Line2D
    handles, labels = [], []
    shown = set()
    for group_name, group_keys in palette_groups.items():
        present = [k for k in group_keys if k in [str(g) for g in unique_groups]]
        if not present:
            continue
        handles.append(Line2D([0], [0], color="none"))
        labels.append(f"── {group_name} ──")
        for k in present:
            c = _lookup_color(colors, k)
            handles.append(Line2D([0], [0], marker="o", color="none",
                                  markerfacecolor=c, markersize=5))
            labels.append(k)
            shown.add(k)
    for g in unique_groups:
        if str(g) not in shown:
            c = _lookup_color(colors, g)
            handles.append(Line2D([0], [0], marker="o", color="none",
                                  markerfacecolor=c, markersize=5))
            labels.append(str(g))
    ax.legend(handles, labels, fontsize=5, markerscale=1,
              bbox_to_anchor=(1.01, 1), loc="upper left")


def _scatter_grouped(coords, grouping, palette, legend, ax, title,
                     palette_groups=None):
    ax = ax or plt.gca()
    groups = grouping.values
    unique = pd.unique(groups)
    colors = (palette if isinstance(palette, dict)
              else dict(zip(unique, sns.color_palette("tab20", len(unique)))))
    for g in unique:
        mask = groups == g
        c    = _lookup_color(colors, g) if isinstance(colors, dict) else "#888888"
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[c], s=4, alpha=0.6, label=str(g), linewidths=0)
    if legend != "off" and legend is not False:
        if palette_groups and isinstance(colors, dict):
            _draw_grouped_legend(ax, unique, colors, palette_groups)
        else:
            ax.legend(fontsize=5, markerscale=2,
                      bbox_to_anchor=(1.01, 1), loc="upper left")
    if title:
        ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


def pca_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
             n_components=2, palette_groups=None):
    X   = StandardScaler().fit_transform(exp_df.fillna(0).values)
    pca = PCA(n_components=n_components, random_state=42)
    coords   = pca.fit_transform(X)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    ax = _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=7)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=7)
    return ax


def umap_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
              n_neighbors=15, min_dist=0.3, palette_groups=None):
    X      = StandardScaler().fit_transform(exp_df.fillna(0).values)
    coords = UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                  random_state=42).fit_transform(X)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    return _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)


def tsne_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
              perplexity=30, palette_groups=None):
    X      = StandardScaler().fit_transform(exp_df.fillna(0).values)
    n_comp = min(50, X.shape[1] - 1, X.shape[0] - 1)
    X_pca  = PCA(n_components=n_comp, random_state=42).fit_transform(X)
    coords = TSNE(n_components=2, perplexity=min(perplexity, len(X) - 1),
                  random_state=42, init="pca").fit_transform(X_pca)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    return _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)

print("Public compatibility shim loaded.")

In [ ]:
import warnings

warnings.filterwarnings("ignore")

# Script from Vitya to stream files to s3 directly

In [ ]:
#!/usr/bin/env python3
"""
Stream a file from an HTTPS URL directly into S3 using boto3 (no local disk).
Assumes AWS creds/region are already configured in ~/.aws.

Requirements:
  pip install boto3 requests

Example:
  python https_to_s3_stream.py --url "https://example.com/bigfile.bin" --bucket "my-bucket" --key "uploads/bigfile.bin"
"""

import argparse
import sys
from typing import Optional

import boto3
import requests
from boto3.s3.transfer import TransferConfig


class Progress:
    def __init__(self, total: Optional[int]) -> None:
        self.total = total
        self.sent = 0

    def __call__(self, bytes_amount: int) -> None:
        self.sent += bytes_amount
        if self.total:
            pct = (self.sent / self.total) * 100.0
            sys.stderr.write(f"\rUploaded {self.sent}/{self.total} bytes ({pct:5.1f}%)")
        else:
            sys.stderr.write(f"\rUploaded {self.sent} bytes")
        sys.stderr.flush()


def stream_https_to_s3(url: str, bucket: str, key: str) -> None:
    s3 = boto3.client("s3")  # uses ~/.aws credentials + config automatically

    with requests.get(url, stream=True, allow_redirects=True, timeout=(10, 300)) as r:
        r.raise_for_status()
        r.raw.decode_content = True

        total = r.headers.get("Content-Length")
        total_int = int(total) if total and total.isdigit() else None

        extra_args = {}
        ct = r.headers.get("Content-Type")
        if ct:
            extra_args["ContentType"] = ct

        config = TransferConfig(
            multipart_threshold=8 * 1024 * 1024,
            multipart_chunksize=8 * 1024 * 1024,
            max_concurrency=4,
            use_threads=True,
        )

        s3.upload_fileobj(
            Fileobj=r.raw,  # <-- stream from HTTPS
            Bucket=bucket,
            Key=key,
            ExtraArgs=extra_args or None,
            Config=config,
            Callback=Progress(total_int),
        )

    sys.stderr.write("\nDone.\n")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True)
    ap.add_argument("--bucket", required=True)
    ap.add_argument("--key", required=True)
    args = ap.parse_args()

    stream_https_to_s3(args.url, args.bucket, args.key)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

# Palettes and general variables

In [ ]:
lymphoma_ontogeny_palette = {
    # --- NORMAL B-CELLS: THE COLD SPECTRUM (Blues, Teals, Greens) ---
    # Primary/Early Development
    'Bone_marrow_CD19+_': 'blue',    # Deepest Navy
    'Immature_': '#08519c',             # Dark Blue
    'Naive_': '#4292c6',                # Medium Blue
    
    # Germinal Center (Cold variants)
    'GC_': '#006d2c',                   # Dark Green
    'Centroblast_': '#41ab5d',          # Medium Green
    'Centrocyte_': '#a1d99b',           # Pale Green
    
    # Mature & Memory
    'B_cells_': 'lawngreen',              # Forest Green
    'MZ_': 'olive',                   # Seafoam Green
    'Memory_': '#00ced1',               # Dark Turquoise
    
    # Terminal Differentiation (Cold Neutrals)
    'Plasmablast_': '#5f9ea0',          # Cadet Blue
    'Plasma_': 'grey',               # Steel Blue-Gray

    # --- MALIGNANT TYPES: THE WARM SPECTRUM (Reds, Oranges, Purples, Black) ---
    # Follicular (The Purple/Magenta family)
    'Follicular_Lymphoma_': '#6a0dad',      # Vivid Purple
    'Follicular_Lymphoma_GCB': '#9400d3',   # Dark Violet
    
    # DLBCL (The Red/Orange family)
    'Diffuse_Large_B_Cell_Lymphoma_ABC': '#ff4500', # Orange-Red
    'Diffuse_Large_B_Cell_Lymphoma_GCB': '#d62728', # Classic Red
    'Diffuse_Large_B_Cell_Lymphoma_': '#8b0000',    # Dark Red
    
    # High-Grade & Burkitt (Deep Contrast)
    'High_Grade_B_Cell_Lymphoma_': '#ff00ff',       # Magenta/Fuchsia
    'Burkitt_Lymphoma_': 'orange'                  # Absolute Black
}

rna_batch_palette = {
    # RNASeq Group (Cooler tones: Blues, Purples, Teals, Greens)
    "RNASeq_FF_Total": "#1f77b4",  # Deep Blue
    "RNASeq_FF_PolyA": "#17becf",  # Bright Cyan
    "RNASeq_FFPE_Exome_capture": "#9467bd",  # Muted Purple
    "RNASeq_FF_rRNADepletion": "#000080",  # Navy
    "RNASeq_FFPE_PolyA": "#8c564b",  # Deep Brown-ish Purple
    "RNASeq_FF_Unknown": "#bcbd22",  # Olive Green
    "RNASeq_FF_Exome_capture": "#2ca02c",  # Forest Green
    # Affymetrix Family (Reds, Oranges, Pinks)
    "GPL96+97_FF_Unknown": "#d62728",  # Bright Red
    "GPL570_FF_Unknown": "#ff7f0e",  # Vivid Orange
    "GPL570_Unknown_Unknown": "#ffbb78",  # Pale Orange
    "GPL570_FFPE_Unknown": "#dbdb8d",  # Pale Olive
    "GPL96_FF_Unknown": "#f7b6d2",  # Pale Pink
    "GPL96_FFPE_Unknown": "#e377c2",  # Pink
    "GPL6244_FF_Unknown": "#e9967a",  # Dark Salmon
    "GPL6244_FFPE_Unknown": "#fa8072",  # Salmon
    "GPL13938_FFPE_Unknown": "#800000",  # Maroon
    # Agilent Family (Blues, Teals, Lavenders, Grays)
    "GPL1708_FF_Unknown": "#7f7f7f",  # Medium Gray
    "GPL14951_FFPE_Unknown": "#c49c94",  # Pale Brown
    "GPL17077_FF_Unknown": "#c5b0d5",  # Pale Lavender
    "GPL13158_FF_Unknown": "#ad494a",  # Dusty Red
    "GPL17586_FF_Unknown": "#008b8b",  # Dark Cyan
    "GPL17047_FF_Unknown": "#4682b4",  # Steel Blue
    "GPL887_FFPE_Unknown": "#b0c4de",  # Light Steel Blue
    # Illumina Family (Greens)
    "GPL10739_FF_Unknown": "#9edae5",  # Pale Blue (Illumina original)
    "GPL8432_FFPE_Unknown": "#006400",  # Dark Green
    "GPL20188_FF_Unknown": "#556b2f",  # Dark Olive Green
    "GPL23541_FF_Unknown": "#8fbc8f",  # Dark Sea Green
    # Other / Miscellaneous (Neutrals)
    "GPL16686_FF_Unknown": "#333333",  # Near Black
    "GPL26356_FF_Unknown": "#a0522d",  # Sienna
}
cohort_palette = {
    "PUB_DLBCL_CARTFARAMAND": "#f87d06",
    "PUB_DLBCL_JIANGGSE147986": "#1482b7",
    "PUB_DLBCL_GOYA": "#b72114",
    "PUB_DLBCL_NCICCR": "#b7b014",
    "PUB_DLBCL_PARKSRP100394": "#5bd66f",
    "PUB_DLBCL_SHISRP312586": "#2306f8",
    "PUB_DLBCL_YANCART": "#d67e5b",
    "PUB_FL_DAVE": "#0626f8",
    "PUB_FL_GSE62241": "#b71481",
    "PUB_FL_HUET": "#14b75d",
    "PUB_FL_PARSA": "#146db7",
    "PUB_FL_GUO": "#a3d65b",
    "PUB_FL_GSE148070": "#5b95d6",
    "PUB_FL_BRODTKORB": "#3514b7",
    "PUB_FL_CIBERSORTX": "#9ff806",
    "PUB_FL_HORN": "#5bd6cb",
    "PUB_FL_PASTORE": "#d66f5b",
    "PUB_FL_LEBRUN": "#b71496",
    "PUB_FL_ORICCHIO": "#9d5bd6",
    "PUB_FL_TASKINEN": "#14b772",
    "PUB_FL_DREYLING": "#8d5bd6",
    "PUB_FL_NCIFFPEREBUILT": "#f80625",
    "PUB_FL_NCISTAUDT": "#f80682",
    "PUB_FL_TOBIN": "#069ef8",
    "PUB_DLBCL_GSE11318": "#f80644",
    "PUB_DLBCL_BAGS": "#5bd6ac",
    "PUB_DLBCL_ETABM346": "#5bd660",
    "PUB_DLBCL_DLC1": "#5bd6bb",
    "PUB_DLBCL_NCIACALA": "#06f860",
    "PUB_DLBCL_GSE10846": "#5b85d6",
    "PUB_DLBCL_GSE117556": "#4914b7",
    "PUB_DLBCL_GSE19246": "#9b06f8",
    "PUB_DLBCL_GSE22898": "#c2d65b",
    "PUB_DLBCL_GSE23501": "#bef806",
    "PUB_DLBCL_GSE31312": "#b73614",
    "PUB_DLBCL_GSE32918": "#f706f8",
    "PUB_DLBCL_GSE34171": "#5e14b7",
    "PUB_DLBCL_GSE64555": "#d65bb3",
    "PUB_DLBCL_GSE87371": "#06dbf8",
    "PUB_DLBCL_GSE93984": "#d65ba3",
    "PUB_DLBCL_LYMPHOMICS": "#d65b84",
    "COLL_DLBCL_UMLOSSOSTRANSL": "#5b75d6",
    "CRON3": "#06f8f6",
    "E-MTAB-3974": "#f806da",
    "GSE107683": "#5bd1d6",
    "GSE12195": "#f89c06",
    "GSE12366": "#0607f8",
    "GSE12453": "#b2d65b",
    "GSE12845": "#66d65b",
    "GSE13411": "#14b787",
    "GSE136248": "#4206f8",
    "GSE136249": "#1bb714",
    "GSE141005": "#b906f8",
    "GSE144089": "#f80663",
    "GSE149089": "#f85e06",
    "GSE15271": "#aab714",
    "GSE156751": "#14b748",
    "GSE17186": "#1458b7",
    "GSE19599": "#b78614",
    "GSE22886": "#d806f8",
    "GSE36907": "#bc5bd6",
    "GSE38697": "#b79b14",
    "GSE46261": "#45b714",
    "GSE51528": "#1444b7",
    "GSE55267": "#1497b7",
    "GSE56314": "#b714ab",
    "GSE61912": "#76d65b",
    "GSE64028": "#06f841",
    "GSE68878": "#d68e5b",
    "GSE69033": "#06f822",
    "GSE85543": "#d2d65b",
    "GSE89282": "#06bdf8",
    "GSE99635": "#30b714",
    "Leandro-Normals": "#af14b7",
    "PUB_Beguelin_GSE45982": "#d6ca5b",
    "PUB_Health_B_cells_GSE110669": "#d69e5b",
    "PUB_Health_B_cells_GSE111310": "#b7141c",
    "PUB_Health_B_cells_GSE115655": "#b71431",
    "PUB_Health_B_cells_GSE119234": "#06f87f",
    "PUB_Health_B_cells_GSE139833": "#59b714",
    "PUB_Health_B_cells_GSE154583": "#5b66d6",
    "PUB_Health_B_cells_GSE92387": "#f8bb06",
    "PUB_Health_Bunting_GSE84022": "#d65b94",
    "PUB_Pastore_GSE119103": "#ac5bd6",
    "PUB_Suntsova_GSE120795": "#80f806",
    "Kassandra": "#14b79c",
    "SOM": "#61f806",
    "MDA_Strati_FL": "#2014b7",
}

platform_palette = {
    # Standalone / Sequencing (Indigo/Purple Spectrum)
    "RNASeq": "#4B0082",  # Indigo
    "RNAseq": "#4B0082",  # Matches RNASeq
    "Kassandra": "#6A5ACD",  # SlateBlue (Deconvolution/Virtual)
    # Affymetrix Group (Oranges & Reds)
    "GPL96+97": "#FF4500",  # OrangeRed
    "GPL96+GPL97": "#FF4500",  # Synonymous with GPL96+97
    "GPL570": "#FF8C00",  # DarkOrange
    "GPL96": "#B22222",  # FireBrick
    "GPL6244": "#E9967A",  # DarkSalmon (HuGene 1.0 ST)
    "GPL13938": "#CD5C5C",  # IndianRed
    "A-GEOD-19803": "#A52A2A",  # Brown/Dark Red (Affy Gene 1.0 ST)
    # Agilent Group (Blues & Teals)
    "GPL1708": "#008080",  # Teal
    "GPL14951": "#4682B4",  # SteelBlue
    "GPL17077": "#00CED1",  # DarkTurquoise
    "GSE17077": "#00CED1",  # Matches GPL17077 (likely a typo in source)
    "GPL13158": "#1E90FF",  # DodgerBlue
    "GPL17586": "#20B2AA",  # LightSeaGreen
    "GPL17047": "#5F9EA0",  # CadetBlue
    "GPL887": "#B0C4DE",  # LightSteelBlue
    # Illumina Group (Greens)
    "GPL10739": "#2E8B57",  # SeaGreen
    "GPL10739+GPL19251": "#3CB371",  # MediumSeaGreen
    "GPL20188": "#6B8E23",  # OliveDrab
    "GPL23541": "#8FBC8F",  # DarkSeaGreen
    "GPL8432": "#006400",  # DarkGreen
    # Other / Miscellaneous (Earth Tones & Grays)
    "GPL16686": "#8B4513",  # SaddleBrown
    "GPL26356": "#A0522D",  # Sienna
    float("nan"): "#D3D3D3",  # LightGray (for missing values)
    None: "#D3D3D3",  # Safety for None types
}

portrait_palette = {
    "Immune-Enriched, Fibrotic": "#009999",
    "Immune-Enriched, Non-fibrotic": "#0000CC",
    "Fibrotic": "#CC0033",
    "Immune Desert": "#FF9900",
    "IE/F": "#009999",
    "IE": "#0000CC",
    "F": "#CC0033",
    "D": "#FF9900",
}
primary_or_metastatic_palette = {"P": "#1f77b4", "M": "#d62728"}
age_palette = {"low": "#1f77b4", "mid": "#ff7f0e", "high": "#d62728"}
gender_palette = {
    "M": "#1f77b4",
    "F": "#e377c2",
}
treatment_palette = {"TKI": "#1f77b4", "IO+TKI": "#2ca02c"}
response_palette = {"CR": "#2ecc71", "PR": "#3498db", "SD": "#f1c40f", "PD": "#e74c3c"}

In [ ]:
download_folder = os.path.join(FL_DATA_ROOT, "cohorts")
os.makedirs(download_folder, exist_ok=True)

# download_folder = '<some other local path>'  # OR set custom path
files = ["expressions.tsv"]

download_folder

In [ ]:
def plot_distinct_ontogeny_legend(palette):
    groups = {
        "Developmental: Early to GC": [
            'Bone_marrow_CD19+_', 'Immature_', 'Naive_', 
            'GC_', 'Centroblast_', 'Centrocyte_'
        ],
        "Developmental: Mature to Plasma": [
            'B_cells_', 'MZ_', 'Memory_', 
            'Plasmablast_', 'Plasma_' # Now Brown/Bronze
        ],
        "Malignancy: Follicular": [
            'Follicular_Lymphoma_', 'Follicular_Lymphoma_GCB'
        ],
        "Malignancy: DLBCL & Aggressive": [
            'Diffuse_Large_B_Cell_Lymphoma_ABC', 'Diffuse_Large_B_Cell_Lymphoma_GCB', 
            'Diffuse_Large_B_Cell_Lymphoma_', 'High_Grade_B_Cell_Lymphoma_', 
            'Burkitt_Lymphoma_' # Now Vivid Reds/Black
        ]
    }
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    handles = []
    for group_name, items in groups.items():
        handles.append(mpatches.Patch(color='white', label=f"--- {group_name} ---"))
        for item in items:
            if item in palette:
                handles.append(mpatches.Patch(color=palette[item], label=item))
    
    leg = ax.legend(
        handles=handles, loc='center', ncol=2, 
        fontsize=10, frameon=True, 
        title="Distinct B-Cell Ontogeny & Lymphoma Legend",
        title_fontsize=12, labelspacing=0.7
    )
    
    for text in leg.get_texts():
        if "---" in text.get_text():
            text.set_fontweight('bold')
            
    plt.tight_layout()
    plt.show()

plot_distinct_ontogeny_legend(lymphoma_ontogeny_palette)

# the internal cohort database cohorts

Google drive https://docs.google.com/spreadsheets/d/1tEJy9ArUGKGIeGXU9CbD9WreGlgvpRk_PCJmnOo0lL4/edit?gid=406586184#gid=406586184



In [ ]:
cohorts_list = """PUB_DLBCL_CARTFARAMAND
PUB_DLBCL_JIANGGSE147986
PUB_DLBCL_GOYA
PUB_DLBCL_NCICCR
PUB_DLBCL_PARKSRP100394
PUB_DLBCL_SHISRP312586
PUB_DLBCL_YANCART
PUB_FL_DAVE
PUB_FL_GSE62241
PUB_FL_HUET
PUB_FL_PARSA
PUB_FL_GUO
PUB_FL_GSE148070
PUB_FL_BRODTKORB
PUB_FL_CIBERSORTX
PUB_FL_HORN
PUB_FL_PASTORE
PUB_FL_LEBRUN
PUB_FL_ORICCHIO
PUB_FL_TASKINEN
PUB_FL_DREYLING
PUB_FL_NCIFFPEREBUILT
PUB_FL_NCISTAUDT
PUB_FL_TOBIN
PUB_DLBCL_GSE11318
PUB_DLBCL_BAGS
PUB_DLBCL_ETABM346
PUB_DLBCL_GSE38202
PUB_DLBCL_DLC1
PUB_DLBCL_NCIACALA
PUB_DLBCL_GSE10846
PUB_DLBCL_GSE117556
PUB_DLBCL_GSE19246
PUB_DLBCL_GSE22898
PUB_DLBCL_GSE23501
PUB_DLBCL_GSE31312
PUB_DLBCL_GSE32918
PUB_DLBCL_GSE34171
PUB_DLBCL_GSE64555
PUB_DLBCL_GSE87371
PUB_DLBCL_GSE93984
PUB_DLBCL_GSE98588
PUB_DLBCL_LYMPHOMICS
COLL_DLBCL_UMLOSSOSTRANSL""".split(
    "\n"
)
cohorts_list.append("CRON3")
cohorts_list

## Files download

You can use the following filters for your request

The full filter schema was documented on an internal wiki; the fields it accepts are the ones shown below.

```
filters = {
'COHORT_ID!' : ['CRON3'],
# 'S3_COHORT_ID': [None], 
# 'PUBLIC_PATIENT_ID' : [None], 
# 'DIAGNOSIS!' : ["Pancreatic_Ductal_Adenocarcinoma", "Pancreatic"],
'TUMOR_NORMAL'  : ["Tumor"], 
'CASE_TYPE_DNA' : [True],
'CASE_TYPE_RNA' : [True],
# 'PRIMARY_OR_METASTATIC' : ["P"], 
# 'SITE_OF_METASTASIS' : [None], 
# 'GRADE' : [None],
# 'PLATFORM_RNA' : ["RNAseq"],
# 'RNASEQ_SOURCE' : ["FFPE"], 
# 'RNASEQ_LIBRARY_TYPE'  : ["PolyA"], 
# 'RNASEQ_CALC_BY_BG'  : [True], 
# 'RNA_BATCH' : ["FF_PolyA"], 
# 'DNA_TYPE'  : ["WES"],
# 'DNA_CALC_BY_BG'  : [True], 
# 'OS' : [None],
# 'PFS' : [None],
# 'DFS' : [None],
# 'OS_FLAG' : [None],
# 'PFS_FLAG' : [None],
# 'DFS_FLAG' : [None],
# 'NEO_CHEMO_THERAPY' : [None],
# 'NEO_THERAPY' : [None],
# 'ADJ_CHEMO_THERAPY' : [None],
# 'ADJ_THERAPY' : [None],
# 'TREATMENT_REGIMEN' : [None],
# 'RESPONSE' : [None],
# 'RESPONSE_TYPE' : [None],
# 'PIPELINES_VERSION' : ["OLD"], 
# 'AUTHOR' : [None]


}

```

In [ ]:
for cohort_name in tqdm(cohorts_list[40:]):
    download_folder_current = os.path.join(
        os.path.join(FL_DATA_ROOT, "cohorts"), cohort_name
    )
    os.makedirs(download_folder, exist_ok=True)

    # The download below ran against an internal cohort database via its Python
# client. That database is not public; DATA_AVAILABILITY.md lists every cohort
# it served with the accession or application route for each. The call is kept,
# commented, as the record of what was fetched and with which filters.
# await cohort_tools.download_samples(
#         download_dir=download_folder_current,  # set the download directory
#         filters={"COHORT_ID": cohort_name},  # define the samples IDs, optional
#         files=files,
#         expand_annotation=True,
#     )

### Adding Chronos

In [ ]:
cohort_name = "CRON3"
download_folder_current = os.path.join(
    os.path.join(FL_DATA_ROOT, "cohorts"), cohort_name
)
os.makedirs(download_folder_current, exist_ok=True)

# The download below ran against an internal cohort database via its Python
# client. That database is not public; DATA_AVAILABILITY.md lists every cohort
# it served with the accession or application route for each. The call is kept,
# commented, as the record of what was fetched and with which filters.
# await cohort_tools.download_samples(
#     download_dir=download_folder_current,  # set the download directory
#     filters={
#         "COHORT_ID": cohort_name,
#         "DIAGNOSIS": [
#             "Follicular_Lymphoma",
#             "Diffuse_Large_B_Cell_Lymphoma",
#             "Burkitt_Lymphoma",
#         ],
#     },  # define the samples IDs, optional
#     files=files,
#     expand_annotation=True,
# )

In [ ]:
# NO OPT OUT YES SAMPLES, TAKE THEM ALL!!!
cohort = "CRON3"
ann = pd.read_csv(
    f"$FL_DATA_ROOT/cohorts/{cohort}/annotation.tsv",
    sep="\t",
)
ann.dropna(axis=1)

In [ ]:
1

In [ ]:
1

## Expressions and annotations read and join

In [ ]:
ann_df_list = []
for cohort in tqdm(cohorts_list):
    ann_df_list.append(
        pd.read_csv(
            f"$FL_DATA_ROOT/cohorts/{cohort}/annotation.tsv",
            sep="\t",
        )
    )
ann = pd.concat(ann_df_list)

exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "cohorts", "expressions.tsv"),
    sep="\t",
    index_col=0,
)
exp = exp[(~exp.isna()).sum()[(~exp.isna()).sum() > 5000].index]  # .fillna(0)

ann = ann[ann["SAMPLE_ID"] == exp.index]
exp = exp.reindex(ann["SAMPLE_ID"])

ann[
    [
        "PUBLIC_SAMPLE_LABEL",
        "PUBLIC_PATIENT_ID",
        "COHORT_LABEL",
        "SAMPLE_NAME",
        "SAMPLE_ID",
    ]
]

In [ ]:
exp

In [ ]:
sum(ann["SAMPLE_ID"] != exp.index)

In [ ]:
ann[ann["SAMPLE_NAME"] + "-" + ann["COHORT_LABEL"] != exp.index][
    "COHORT_LABEL"
].unique()

In [ ]:
print((~ann.isna()).sum().sort_values().to_string())

In [ ]:
exp_df_list = []
for cohort in tqdm(cohorts_list):
    exp_df_list.append(
        read_dataset(
            os.path.join(FL_DATA_ROOT, "cohorts", cohort, "expressions.tsv")
        )
    )
exp = pd.concat(exp_df_list)
# exp.to_csv(
#    os.path.join(FL_DATA_ROOT, "cohorts", "expressions.tsv"), sep="\t"
# )
exp = exp[(~exp.isna()).sum()[(~exp.isna()).sum() > 5000].index]  # .fillna(0)
exp

In [ ]:
exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "cohorts", "expressions.tsv"),
    sep="\t",
    index_col=0,
)
exp = exp[(~exp.isna()).sum()[(~exp.isna()).sum() > 5000].index]  # .fillna(0)
exp

### Expressions and annotation to the same order

In [ ]:
ann = ann[ann["SAMPLE_ID"] == exp.index]
ann

In [ ]:
exp = exp.reindex(ann["SAMPLE_ID"])
exp

In [ ]:
print((~exp.isna()).sum().sort_values().to_string())

In [ ]:
### Missing features distribution

(~exp.isna()).sum().sort_values().reset_index().plot()

### Normal_B_cells addition

In [ ]:
def import_data_all_vs_one(target_cell_type, ann=ann, test_cohorts=test_cohorts):
    """
    Imports annotation and expression data to train 1 vs other BAGS model for target cell type
    :param target_cell_type: selected cell type
    :param ann: annotation, must contain Cell_type column
    :return X: train features dataframe
    :return y: train labels series
    :return x: annotation
    :return holdouts: holdout cohorts for feature selection
    :return cell_to_class: dict to convert cell types to class labels
    :return class_to_cell: dict to convert labels to cell types
    :return all_exp: overall expressions
    :return ann: annotation again, yes, its a bit idiotic, will be corrected
    """

    ann.Cell_type = ann.Cell_type.str.replace("Plasmablast", "Plasma")
    ann.Cell_type = ann.Cell_type.str.replace("PreBI", "ProB")
    ann.Cell_type = ann.Cell_type.str.replace("PreBII", "PreB")
    ann.Source_tissue = ann.Source_tissue.str.replace("PBMC", "PB")
    ann = ann[
        ann.Source_tissue.isin(["LN", "Tonsil", "PB", "Spleen", "BM", "Intestine"])
    ]

    cell_to_class = {k: 0 for k in cell_types if k != target_cell_type}
    cell_to_class[target_cell_type] = 1

    tmp_ctc = (
        pd.DataFrame(pd.Series(cell_to_class))
        .reset_index()
        .rename(columns={"index": "cell_type", 0: "cell_class"})
        .groupby(["cell_class"])
        .aggregate({"cell_type": lambda x: "/".join(x)})
        .reset_index()
    )
    class_to_cell = dict(zip(tmp_ctc.cell_class, tmp_ctc.cell_type))
    poor_cohorts = ["GSE12845", "GSE141005", "GSE89282", "GSE13411"] + ["GSE64028"]

    cohorts_to_exclude = poor_cohorts + test_cohorts

    ann = ann[ann.Cohort.apply(lambda x: x not in cohorts_to_exclude)]
    ann = ann[ann.Cell_type.isin(list(cell_to_class.keys()))]
    print(ann.Cell_type.value_counts())

    all_exp = []
    for i in tqdm_notebook(glob("${FL_SHARED_ROOT}/projects/FL_META/Normal_B_cells/training_data/*.tsv")):
        c_ds = os.path.splitext(os.path.split(i)[1])[0]
        if c_ds in ann.Cohort.unique().tolist():
            print(c_ds)
            tmp_ann = ann[ann.Cohort == c_ds]
            all_exp.append(
                read_dataset(i)
                .rename_axis(["Gene"])
                .reset_index()
                .drop_duplicates(subset="Gene")
                .set_index("Gene")[tmp_ann.index & read_dataset(i).columns]
            )  # selecet only samples from common annotation
    all_exp = pd.concat(all_exp, axis=1)
    all_exp = all_exp.dropna()
    # display(all_exp.head())

    X = all_exp[all_exp.T.index & ann.index].T  # .rank(axis=1)
    y = ann.Cell_type.loc[X.index].map(cell_to_class)
    x = ann.loc[y.index]
    holdouts = x.Cohort.unique()

    return (X, y, x, holdouts, cell_to_class, class_to_cell, all_exp, ann)

In [ ]:
files = !ls ${FL_SHARED_ROOT}/projects/FL_META/Normal_B_cells/training_data/
files

In [ ]:
files = !ls ${FL_SHARED_ROOT}/projects/FL_META/Normal_B_cells/training_data/

ann_normal = read_dataset('bcell_typing_train_data.tsv')
exp_df_list_normal = []
for file in tqdm(files):
    if file != 'GSE85289.tsv':
        exp_normal_local =  read_dataset(
                f"${FL_SHARED_ROOT}/projects/FL_META/Normal_B_cells/training_data/{file}"
            ).rename_axis(["Gene"]).reset_index().drop_duplicates(
                subset="Gene").set_index("Gene")
        exp_df_list_normal.append(exp_normal_local)
exp_normal = pd.concat(exp_df_list_normal, axis=1).dropna(axis=0)
exp_normal = exp_normal[exp_normal.columns.intersection(ann_normal.index)]
ann_normal = ann_normal.reindex(exp_normal.columns)
exp_normal

In [ ]:
# ann_normal = read_dataset('${FL_SHARED_ROOT}/projects/FL_META/Normal_B_cells/Normal_B_cells_annotation_250321.tsv')
ann_normal = read_dataset("bcell_typing_train_data.tsv")
ann_normal

In [ ]:
supervenn(
    [set(ann_normal.index), set(exp_normal.columns)],
    ["ann_normal.index", "exp_normal.columns"],
)

In [ ]:
exp_normal

In [ ]:
comb_ann.to_csv(os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t")

### B cells from Kassandra

In [ ]:
kassandra_ann = pd.read_csv(
    "${FL_SHARED_ROOT}/Deconvolution/product/models/tumor_model_v15_4/cells_annot.tsv", sep="\t"
)
kassandra_ann = kassandra_ann[kassandra_ann["Cell_type"].str.contains("B_")]


kassandra_exp = pd.read_csv(
    "${FL_SHARED_ROOT}/Deconvolution/product/models/tumor_model_v15_4/cells_expr.tsv", sep="\t"
).set_index("Gene")
kassandra_exp = kassandra_exp[kassandra_ann["Sample"]]
kassandra_exp

In [ ]:
comb_exp = pd.concat([exp, exp_normal.T, kassandra_exp.T], axis=0)  # .dropna(axis=1)
comb_exp

In [ ]:
comb_ann = pd.concat(
    [ann.set_index("SAMPLE_ID"), ann_normal, kassandra_ann.set_index("Unnamed: 0")],
    axis=0,
)
comb_ann

In [ ]:
comb_exp.to_csv(os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t")

In [ ]:
comb_ann.to_csv(os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t")

In [ ]:
kassandra_ann

### Add expressions from Arsen

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann = comb_ann[comb_ann.Cohort_general != "SOM"]
# comb_ann["Cohort_general"] = comb_ann["COHORT_LABEL"]
# comb_ann["Cohort_general"][comb_ann["COHORT_LABEL"].isna()] = "Kassandra"
# comb_ann["Cohort_general"][~comb_ann["Cohort"].isna()] = comb_ann["Cohort"][
#    ~comb_ann["Cohort"].isna()
# ]

In [ ]:
comb_ann = comb_ann[comb_ann.Cohort_general != "SOM"]
comb_ann

#### Python code

In [ ]:
import pandas as pd
import pyreadr
from biomart import BiomartServer

result = pyreadr.read_r("DLBCL_selected_samples.Rdata")  # also works for Rds

# 1. Connect to Ensembl BioMart
server = BiomartServer("http://www.ensembl.org/biomart")
mart = server.datasets["hsapiens_gene_ensembl"]

# 2. Define your list of probe IDs
probe_ids = result["indata"].index.to_list()
mapping_list = []

for i in tqdm(range(0, len(probe_ids) // 100)):
    if i != (len(probe_ids) // 100) - 1:
        probe_ids_local = probe_ids[i * 100 : (i + 1) * 100]
    else:
        probe_ids_local = probe_ids[i * 100 :]
    # 3. Query the mapping
    response = mart.search(
        {
            "filters": {"affy_hg_u133_plus_2": probe_ids_local},
            "attributes": ["affy_hg_u133_plus_2", "hgnc_symbol", "entrezgene_id"],
        }
    )

    # 4. Convert to DataFrame
    probes_mapping = pd.read_csv(
        io.StringIO(response.text),
        sep="\t",
        header=None,
        names=["ProbeID", "HGNC", "Entrez"],
    )
    mapping_list.append(probes_mapping)
    print(len(probes_mapping))
probes_mapping_all = pd.concat(mapping_list)
probes_mapping_all.to_csv("GPL570_mapping_probes.csv")

In [ ]:
result = pyreadr.read_r("DLBCL_selected_samples.Rdata")  # also works for Rds
result["indata"]

In [ ]:
mapping_probes = pd.read_csv("GPL570_mapping_probes.csv")[["ProbeID", "HGNC"]].dropna()
mapping_probes

In [ ]:
arsen_exp_with_genes = (
    result["indata"]
    .reset_index()
    .merge(mapping_probes, how="left", left_on="index", right_on="ProbeID")
    .dropna(subset=["ProbeID", "HGNC"])
    .drop(["ProbeID", "index"], axis=1)
)

arsen_exp_with_genes = arsen_exp_with_genes.groupby("HGNC").mean()
arsen_exp_with_genes

In [ ]:
sns.histplot(
    data=arsen_exp_with_genes,
    # The original plotted one specific sample from a proprietary cohort; the
    # identifier is withheld (see DATA_AVAILABILITY.md). Any column serves for
    # this distribution check.
    x=arsen_exp_with_genes.columns[0],
    bins=30,
    # color="#555555",
    # edgecolor='white',
    linewidth=0.1,
)

In [ ]:
ann_arsen.Typ_Grob.value_counts()

In [ ]:
ann_arsen = result["pData"]
ann_arsen["Diagnosis_cell_type"] = ann_arsen["Typ_Grob"].map(
    {
        "DLBCL": "Diffuse_Large_B_Cell_Lymphoma",
        "FL": "Follicular_Lymphoma",
        "BL": "Burkitt_Lymphoma",
        "FL/DLBCL": "Follicular_Lymphoma",
        "normal control": "B_cells",
        "other": "Other",
    }
)
ann_arsen["Diagnosis_cell_type_general"] = ann_arsen["Typ_Grob"].map(
    {
        "DLBCL": "Diffuse_Large_B_Cell_Lymphoma",
        "FL": "Follicular_Lymphoma",
        "BL": "Burkitt_Lymphoma",
        "FL/DLBCL": "Follicular_Lymphoma",
        "normal control": "B_cells",
        "other": "Other",
    }
)

ann_arsen["PLATFORM_RNA"] = "GPL570"
ann_arsen["RNA_BATCH"] = "GPL570_Unknown_Unknown"
ann_arsen["SAMPLE_NAME"] = ann_arsen["MPINR"]
ann_arsen["TUMOR_NORMAL"] = "Tumor"
ann_arsen["TUMOR_NORMAL"][ann_arsen["Typ_Grob"] == "normal control"] == "Normal"
ann_arsen["PUBLIC_PATIENT_ID"] = ann_arsen["MPINR"]
ann_arsen["COHORT_LABEL"] = "SOM"
ann_arsen["DIAGNOSIS"] = ann_arsen["Typ_Grob"].map(
    {
        "DLBCL": "Diffuse_Large_B_Cell_Lymphoma",
        "FL": "Follicular_Lymphoma",
        "BL": "Burkitt_Lymphoma",
        "FL/DLBCL": "Follicular_Lymphoma",
        "normal control": "B_cells",
        "other": "Other",
    }
)
ann_arsen["Platform_general"] = "GPL570"
ann_arsen

In [ ]:
comb_ann = comb_ann[comb_ann["COHORT_LABEL"] != "SOM"]
comb_ann

In [ ]:
pd.concat([comb_exp, arsen_exp_with_genes.T]).to_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
)

In [ ]:
pd.concat([comb_ann, ann_arsen]).to_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
)

In [ ]:
comb_exp

### Add MDA_Strati_FL

MDA_Strati_FL compose annotation from scratch

In [ ]:
ann_strati = pd.read_csv("MDA_Strati_FL_annotation.csv").set_index("RS_ID")
ann_strati

In [ ]:
cohort_strati = dS3Cohort(cohort_id="MDA_Strati_FL")
cohort_strati.expressions

In [ ]:
cohort_strati.expressions

In [ ]:
pd.concat([comb_exp, cohort_strati.expressions]).to_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
)

In [ ]:
pd.concat([comb_ann, ann_strati]).to_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
)

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")

### Final preparation of the expression and annotation for AnnData

In [ ]:
comb_ann = (
    pd.read_csv(os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t")
    .set_index("Unnamed: 0")
    .dropna(axis=1, how="all")
)

diseases_list = [
    "Follicular_Lymphoma",
    "Diffuse_Large_B_Cell_Lymphoma",
    "Burkitt_Lymphoma",
    "High_Grade_B_Cell_Lymphoma",
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
]
comb_ann.loc[comb_ann.index[comb_ann.Major_group.isna()], ["Major_group"]] = (
    comb_ann.loc[comb_ann.index[comb_ann.Major_group.isna()],][
        "Diagnosis_cell_type_general"
    ]
)
comb_ann.loc[comb_ann.index[comb_ann.Cohort_general.isna()], ["Cohort_general"]] = (
    comb_ann.loc[comb_ann.index[comb_ann.Cohort_general.isna()],]["COHORT_LABEL"]
)
comb_ann.loc[comb_ann.index[comb_ann.COHORT_LABEL.isna()], ["COHORT_LABEL"]] = (
    comb_ann.loc[comb_ann.index[comb_ann.COHORT_LABEL.isna()],]["Cohort_general"]
)
comb_ann.loc[
    comb_ann.index[
        (comb_ann.TUMOR_NORMAL.isna())
        & (comb_ann.Diagnosis_cell_type_general.isin(diseases_list))
    ],
    ["TUMOR_NORMAL"],
] = "Tumor"
comb_ann.loc[
    comb_ann.index[
        (comb_ann.TUMOR_NORMAL.isna())
        & (~comb_ann.Diagnosis_cell_type_general.isin(diseases_list))
    ],
    ["TUMOR_NORMAL"],
] = "Normal"

comb_ann["SAMPLE_NAME"] = comb_ann.index

# 1. Define specific index lists (Sample names)
# These four lists named individual samples from proprietary cohorts, so the
# identifiers are withheld (see DATA_AVAILABILITY.md). They are kept as empty lists
# rather than deleted: the masks below still reference them, and the RNA_BATCH value
# each one encoded is recorded per sample in the deposited annotation table.
poly_a_indices = []      # 2 samples withheld -> RNASeq_FF_PolyA
exome_ff_indices = []    # 2 samples withheld -> RNASeq_FF_Exome_capture
total_indices = []       # 4 samples withheld -> RNASeq total RNA
unknown_indices = []     # 9 samples withheld -> RNASeq platform unknown

# 2. Assign values using combined masks
# RNASeq_FF_PolyA
mask_poly_a = (
    comb_ann.SAMPLE_NAME.str.startswith(
        ("210514_NovaA", "RNASeq-normal-201", "RNASeq-normal-210108-NovaB_")
    )
    | comb_ann.index.isin(poly_a_indices)
    | comb_ann.Dataset.isin(
        [
            "GSE92387",
            "GSE95282",
            "GSE110417",
            "GSE107011",
            "GSE110999",
            "GSE119103",
            "GSE132835",
            "GSE62408",
        ]
    )
    | comb_ann.Cohort_general.isin(
        [
            "PUB_Health_B_cells_GSE111310",
            "PUB_Health_B_cells_GSE92387",
            "PUB_Health_B_cells_GSE110669",
            "PUB_Beguelin_GSE45982",
            "PUB_Health_B_cells_GSE115655",
            "PUB_Pastore_GSE119103",
        ]
    )
)
comb_ann.loc[mask_poly_a, "RNA_BATCH"] = "RNASeq_FF_PolyA"
comb_ann.loc[mask_poly_a, "PLATFORM_RNA"] = "RNASeq"

# RNASeq_FF_Unknown
mask_unknown = (
    comb_ann.index.isin(unknown_indices)
    | comb_ann.Dataset.isin(["GSE81443", "GSE84022", "GSE100441", "GSE66117"])
    | comb_ann.Cohort_general.isin(
        [
            "PUB_Health_B_cells_GSE139833",
            "Leandro-Normals",
            "PUB_Health_Bunting_GSE84022",
        ]
    )
)
comb_ann.loc[mask_unknown, "RNA_BATCH"] = "RNASeq_FF_Unknown"
comb_ann.loc[mask_unknown, "PLATFORM_RNA"] = "RNASeq"

# RNASeq_FF_rRNADepletion
mask_rrna = comb_ann.Dataset.isin(
    ["GSE118254", "GSE112101"]
) | comb_ann.Cohort_general.isin(
    ["PUB_Health_B_cells_GSE154583", "PUB_Health_B_cells_GSE119234", "PUB_FL_GSE62241"]
)
comb_ann.loc[mask_rrna, "RNA_BATCH"] = "RNASeq_FF_rRNADepletion"
comb_ann.loc[mask_rrna, "PLATFORM_RNA"] = "RNASeq"

# GPL Groupings (Mapping Cohort_general to specific RNA_BATCH values)
gpl_mapping = {
    "GPL13158_FF_Unknown": ["GSE85543"],
    "GPL20188_FF_Unknown": ["GSE68878", "GSE69033"],
    "GPL570_FF_Unknown": [
        "GSE36907",
        "GSE56314",
        "GSE107683",
        "GSE12453",
        "GSE12195",
        "GSE64028",
        "GSE17186",
        "GSE136248",
        "GSE136249",
        "GSE12366",
        "PUB_Suntsova_GSE120795",
        "GSE38697",
        "GSE15271",
        "GSE19599",
        "GSE55267",
    ],
    "GPL23541_FF_Unknown": ["GSE99635"],
    "GPL16686_FF_Unknown": ["GSE89282", "GSE141005", "E-MTAB-3974"],
    "GPL17077_FF_Unknown": ["GSE156751", "GSE144089"],
    "GPL96+97_FF_Unknown": ["GSE22886", "GSE12845"],
    "GPL26356_FF_Unknown": ["GSE149089"],
    "GPL6244_FF_Unknown": ["GSE51528"],
    "GPL17047_FF_Unknown": ["GSE46261"],
    "GPL96_FF_Unknown": ["GSE13411"],
    "GPL10739_FF_Unknown": ["GSE61912"],
}

for batch_val, cohorts in gpl_mapping.items():
    comb_ann.loc[comb_ann.Cohort_general.isin(cohorts), "RNA_BATCH"] = batch_val
    comb_ann.loc[comb_ann.Cohort_general.isin(cohorts), "PLATFORM_RNA"] = (
        batch_val.split("_")[0]
    )

# Remaining single assignments
comb_ann.loc[comb_ann.index.isin(exome_ff_indices), "RNA_BATCH"] = (
    "RNASeq_FF_Exome_capture"
)
comb_ann.loc[comb_ann.index.isin(exome_ff_indices), "PLATFORM_RNA"] = "RNASeq"
comb_ann.loc[comb_ann.index.isin(total_indices), "RNA_BATCH"] = "RNASeq_FF_Total"
comb_ann.loc[comb_ann.index.isin(total_indices), "PLATFORM_RNA"] = "RNASeq"
comb_ann.loc[comb_ann.Cohort_general == "CRON3", "RNA_BATCH"] = (
    "RNASeq_FFPE_Exome_capture"
)
comb_ann.loc[comb_ann.Cohort_general == "CRON3", "PLATFORM_RNA"] = "RNASeq"
comb_ann["PLATFORM_RNA"] = comb_ann["PLATFORM_RNA"].str.replace("RNAseq", "RNASeq")
comb_ann["RNA_BATCH"] = comb_ann["RNA_BATCH"].str.replace(
    "RNAseq_FF_PolyA", "RNASeq_FF_PolyA"
)

comb_ann.loc[
    (comb_ann.RNASEQ_SOURCE.isna() & (comb_ann.RNA_BATCH.str.contains("FFPE"))),
    "RNASEQ_SOURCE",
] = "FFPE"
comb_ann.loc[
    (comb_ann.RNASEQ_SOURCE.isna() & (~comb_ann.RNA_BATCH.str.contains("FFPE"))),
    "RNASEQ_SOURCE",
] = "FF"


comb_ann

In [ ]:
print((~comb_ann_dedup.isna()).sum().sort_values().to_string())

comb_ann.RNASEQ_LIBRARY_TYPE.value_counts()

In [ ]:
pd.crosstab(comb_ann.RNASEQ_LIBRARY_TYPE, comb_ann.RNA_BATCH)

In [ ]:
comb_ann.to_csv(os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t")

#### Classify normal B cell types in the same logic

In [ ]:
print(comb_ann.Diagnosis_cell_type.value_counts().to_string())

In [ ]:
comb_ann[comb_ann.Diagnosis_cell_type == 'B_cells'].T.dropna()

In [ ]:
print(comb_ann[comb_ann.Diagnosis_cell_type == 'B_cells']['Deconvolution_cell_type'].to_string())

In [ ]:
print((~comb_ann[comb_ann.Diagnosis_cell_type == 'B_cells'].isna()).sum().sort_values().to_string())


### Normalize gene expression arrays to NGS-like distributions 

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann = pd.read_csv(
    "comb_ann_unified.csv", sep=","
).set_index("Unnamed: 0")

#comb_ann = pd.read_csv(
#    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
#).set_index("Unnamed: 0")

In [ ]:
def qsmooth(data, groups):
    """
    Python implementation of Smooth Quantile Normalization (qsmooth).

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    groups : pd.Series or list
        The grouping variable (e.g., RNA_BATCH).

    Returns:
    --------
    pd.DataFrame
        Quantile normalized expression data.
    """
    # Ensure groups is a Series and aligned with the data index
    if not isinstance(groups, pd.Series):
        groups = pd.Series(groups, index=data.index)

    # Explicitly align and filter groups to match data rows
    # This prevents IndexError if indices are reordered or have different lengths
    groups = groups.loc[data.index]

    # Important: Reset index to ensure boolean masks align with NumPy arrays
    # NumPy doesn't know about Pandas index alignment, so we use integer-based arrays
    data_values = data.values
    group_values = groups.values
    unique_groups = np.unique(group_values)
    n_samples, n_genes = data.shape

    # 1. Compute individual quantiles for each sample
    # We sort each row to get the distribution
    sorted_data = np.sort(data_values, axis=1)

    # 2. Compute group-specific mean quantiles
    group_quantiles = {}
    for g in unique_groups:
        group_mask = group_values == g
        # Using .values inside the mask ensures we are matching the 0-dimension of sorted_data
        group_quantiles[g] = np.mean(sorted_data[group_mask, :], axis=0)

    # 3. Compute global mean quantiles
    global_quantiles = np.mean(sorted_data, axis=0)

    # 4. Compute the weights (u_q)
    # SST (Total Sum of Squares) at each quantile rank
    sst = np.sum((sorted_data - global_quantiles) ** 2, axis=0)

    # SSW (Within-group Sum of Squares)
    ssw = np.zeros(n_genes)
    for g in unique_groups:
        group_mask = group_values == g
        g_quant = group_quantiles[g]
        ssw += np.sum((sorted_data[group_mask, :] - g_quant) ** 2, axis=0)

    # SSB (Between-group Sum of Squares)
    ssb = sst - ssw

    # Ratio of variability explained by groups
    # Handle division by zero if sst is 0
    u_q = np.divide(ssb, sst, out=np.zeros_like(ssb), where=sst != 0)

    # 5. Smooth the weights
    # The R version uses a GAM. Here we use a rolling window average
    # as a robust approximation for the smoothing spline.
    window_size = max(int(n_genes * 0.05), 11)  # 5% window
    if window_size % 2 == 0:
        window_size += 1

    u_q_series = pd.Series(u_q)
    u_q_smoothed = (
        u_q_series.rolling(window=window_size, center=True, min_periods=1).mean().values
    )

    # 6. Calculate the smoothed quantiles for each group
    # formula: q_smooth = u_q*group_quant + (1-u_q)*global_quant
    norm_quantiles = {}
    for g in unique_groups:
        norm_quantiles[g] = (u_q_smoothed * group_quantiles[g]) + (
            (1 - u_q_smoothed) * global_quantiles
        )

    # 7. Map normalized quantiles back to the original data
    normalized_data = data.copy()

    for i in range(n_samples):
        # Access the group name for the i-th sample
        sample_group = group_values[i]
        sample_values = data_values[i, :]

        # Get the sorted order of the current sample
        order = np.argsort(sample_values)

        # Replace values with the smoothed quantiles based on rank
        # Use .iloc[i, order] to update based on position
        normalized_data.iloc[i, order] = norm_quantiles[sample_group]

    return normalized_data


# Example Usage with your dataframes:
# ----------------------------------
# # Ensure the indices match
# common_samples = comp_exp.index.intersection(comb_ann.index)
# filtered_exp = comp_exp.loc[common_samples].fillna(0) # Handle NaNs if necessary
# batch_groups = comb_ann.loc[common_samples, 'RNA_BATCH']
#
# # Run the normalization
# normalized_exp = qsmooth(filtered_exp, batch_groups)

In [ ]:
(~comb_ann[comb_ann.Major_group == "Other"].isna()).sum().sort_values()

In [ ]:
comb_ann

In [ ]:
comb_ann.Major_group.value_counts().sum()

In [ ]:
platform_mapping = pd.read_csv("platform_classification - sheet1.csv").set_index(
    "Platform ID"
)
platform_mapping_dict = platform_mapping["Broad Classification"].to_dict()
platform_mapping_dict.update({"RNASeq": "Illumina NGS"})
platform_mapping_dict.update({"Kassandra": "Illumina NGS"})

comb_ann["Platform_classified"] = comb_ann["Platform_general"].map(
    platform_mapping_dict
)
comb_ann["Platform_classified"].fillna("na").value_counts()

In [ ]:
pd.DataFrame(comb_ann[["RNA_BATCH", "Platform_classified"]].value_counts())

#### Dataset filtering for normalization

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann = pd.read_csv(
    "comb_ann_unified.csv", sep=","
).set_index("Unnamed: 0")


bad_batches = ["GPL14951_FFPE_Unknown", "GPL8432_FFPE_Unknown", "GPL13938_FFPE_Unknown"]
bad_cohorts = ["SOM"]


# "GPL8432_FFPE_Unknown",
# "GPL6244_FFPE_Unknown",
# "GPL17586_FF_Unknown",
# "GPL20188_FF_Unknown",
# "GPL23541_FF_Unknown",
# "GPL887_FFPE_Unknown",
# "GPL26356_FF_Unknown",
# "GPL17047_FF_Unknown",
# "GPL6244_FF_Unknown",
# "GPL13938_FFPE_Unknown",
# ]
rare_groups = [
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
    "Other",
]


#comb_ann = comb_ann[~comb_ann.RNA_BATCH.isin(bad_batches)]
comb_ann = comb_ann[~comb_ann.Major_group.isin(rare_groups)]
#comb_ann = comb_ann[~comb_ann.COHORT_LABEL.isin(bad_cohorts)]

#comb_ann.loc[
#    comb_ann.Major_group.isin(["Normal_B_cells", "Kassandra"]),
#    ["Major_group"],
#] = "Normal_B_cells"

comb_ann['Diagnosis_with_coo'] = comb_ann.Diagnosis_cell_type_general + '_' + comb_ann['coo_bg'].fillna('')
comb_ann['Diagnosis_unified_with_coo'] = comb_ann.Diagnosis_cell_type_unified + '_' + comb_ann['coo_bg'].fillna('')



#comb_ann.loc[
#    comb_ann.Major_group.isin(["Normal_B_cells", "Kassandra"]),
#    ["Major_group"],
#] = "Normal_B_cells"



common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
filtered_exp = comb_exp.loc[common_samples].dropna(axis=1)
filtered_exp = filtered_exp.loc[~filtered_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
batch_groups = comb_ann.loc[common_samples, "RNA_BATCH"]
batch_groups = batch_groups.loc[~batch_groups.index.duplicated(keep="first"),]
comb_ann_dedup = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]
filtered_exp

In [ ]:
normalized_exp = qsmooth(filtered_exp, batch_groups)
normalized_exp

In [ ]:
comb_ann_dedup

In [ ]:
len(cohorts_counts)

In [ ]:
cohorts_counts = comb_ann_dedup.Cohort_general.fillna("na").value_counts()

GLOBAL_FONT_SIZE = 7.5
FIGURE_SIZE = (30, 20)
BINS = 30

# --- 3. Visualization ---

figure, axs = plt.subplots(
    nrows=8,
    ncols=11,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=FIGURE_SIZE,
)

# Flatten axes for iteration
flat_axs = axs.flatten()

for cohort, ax in zip(cohorts_counts.index, flat_axs):

    # Filter the data
    exp_local = normalized_exp.T[
        comb_ann_dedup.index[comb_ann_dedup.Cohort_general == cohort]
    ].dropna()

    platform = comb_ann_dedup.loc[
        comb_ann_dedup.Cohort_general == cohort, "PLATFORM_RNA"
    ][0]
    batch = comb_ann_dedup.loc[comb_ann_dedup.Cohort_general == cohort, "RNA_BATCH"][0]
    print(cohort, platform, batch)
    if batch == "GPL6244_FFPE_Unknown":
        print(comb_ann_dedup[comb_ann_dedup.RNA_BATCH == batch])

    # if exp_local.max().max() > 50:
    #    exp_local = np.log2(exp_local + 1)

    # Plot histogram
    sns.histplot(
        data=exp_local,
        x=exp_local.columns[0],
        ax=ax,
        bins=BINS,
        # color="#555555",
        # edgecolor='white',
        linewidth=0.1,
    )

    # --- Unified Font Size Customization ---

    # Subplot Title
    ax.set_title(
        f"{cohort} \n {platform} {batch}", fontsize=GLOBAL_FONT_SIZE, fontweight="bold"
    )

    # Axis Labels
    ax.set_xlabel("Gene expression value", fontsize=GLOBAL_FONT_SIZE)
    ax.set_ylabel("Frequency", fontsize=GLOBAL_FONT_SIZE)

    # Tick Labels (X and Y)
    ax.tick_params(axis="both", which="major", labelsize=GLOBAL_FONT_SIZE)

# Adjust layout to prevent overlap
plt.tight_layout()

# --- 4. Saving ---

os.makedirs("plots", exist_ok=True)
save_path = "plots/Dist_qc_1st_sample_in_cohort_qsmooth_no_bad_batches.png"

# plt.savefig(save_path, format="svg", transparent=True, bbox_inches="tight")
plt.savefig(save_path, dpi=300)

plt.show()

In [ ]:
normalized_exp.to_csv(
    "$FL_DATA_ROOT/cohorts/normalized_exp.tsv", sep="\t"
)

In [ ]:
robust_zeroed_exp.to_csv(
    "$FL_DATA_ROOT/cohorts/normalized_zeroed_exp.tsv",
    sep="\t",
)

In [ ]:
def apply_robust_cohort_normalization(
    exp_df, ann_df, cohort_col="Cohort_general", percentile=0.5
):
    """
    Shifts each cohort by a specific percentile and zeros out everything below it.

    Parameters:
    - exp_df: Gene expression DataFrame (Samples x Genes).
    - ann_df: Metadata DataFrame containing cohort info.
    - cohort_col: Column name in ann_df used for grouping.
    - percentile: The percentile (0-100) to use as the new 'zero' baseline.
    """
    normalized_data = exp_df.copy()

    # Align indices to ensure metadata matches expression samples
    common_samples = exp_df.index.intersection(ann_df.index)
    cohort_mapping = ann_df.loc[common_samples, cohort_col]
    unique_cohorts = cohort_mapping.unique()

    print(
        f"Normalizing {len(unique_cohorts)} cohorts using the {percentile}th percentile..."
    )

    for cohort in unique_cohorts:
        # Identify samples in this specific cohort
        samples_in_cohort = cohort_mapping[cohort_mapping == cohort].index
        cohort_slice = normalized_data.loc[samples_in_cohort]

        # Calculate the robust floor (percentile) across the entire cohort block
        # We flatten to 1D to get the percentile of all values combined
        floor_value = np.percentile(cohort_slice.values, percentile)

        # Shift and Clip: Subtract the floor and force negatives to zero
        shifted_data = cohort_slice - floor_value
        normalized_data.loc[samples_in_cohort] = shifted_data.clip(lower=0)

        print(f"Cohort: {cohort:30} | Subtracted Floor: {floor_value:.4f}")

    return normalized_data


# --- Execution ---
# Using 1% as a common threshold for background noise in microarrays/RNASeq
robust_zeroed_exp = apply_robust_cohort_normalization(
    np.log2(normalized_exp + 1),
    comb_ann_dedup,
    cohort_col="Cohort_general",
    percentile=1,  # Adjust this (e.g., 0.1, 0.5, 1.0) based on your noise levels
)

In [ ]:
cohorts_counts = comb_ann_dedup.Cohort_general.fillna("na").value_counts()

GLOBAL_FONT_SIZE = 7.5
FIGURE_SIZE = (30, 20)
BINS = 30

# --- 3. Visualization ---

figure, axs = plt.subplots(
    nrows=8,
    ncols=11,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=FIGURE_SIZE,
)

# Flatten axes for iteration
flat_axs = axs.flatten()

for cohort, ax in zip(cohorts_counts.index, flat_axs):

    # Filter the data
    exp_local = robust_zeroed_exp.T[
        comb_ann_dedup.index[comb_ann_dedup.Cohort_general == cohort]
    ].dropna()

    platform = comb_ann_dedup.loc[
        comb_ann_dedup.Cohort_general == cohort, "PLATFORM_RNA"
    ][0]
    batch = comb_ann_dedup.loc[comb_ann_dedup.Cohort_general == cohort, "RNA_BATCH"][0]
    print(cohort, platform, batch)
    if batch == "GPL6244_FFPE_Unknown":
        print(comb_ann_dedup[comb_ann_dedup.RNA_BATCH == batch])

    # if exp_local.max().max() > 50:
    #    exp_local = np.log2(exp_local + 1)

    # Plot histogram
    sns.histplot(
        data=exp_local,
        x=exp_local.columns[0],
        ax=ax,
        bins=BINS,
        # color="#555555",
        # edgecolor='white',
        linewidth=0.1,
    )

    # --- Unified Font Size Customization ---

    # Subplot Title
    ax.set_title(
        f"{cohort} \n {platform} {batch}", fontsize=GLOBAL_FONT_SIZE, fontweight="bold"
    )

    # Axis Labels
    ax.set_xlabel("Gene expression value", fontsize=GLOBAL_FONT_SIZE)
    ax.set_ylabel("Frequency", fontsize=GLOBAL_FONT_SIZE)

    # Tick Labels (X and Y)
    ax.tick_params(axis="both", which="major", labelsize=GLOBAL_FONT_SIZE)

# Adjust layout to prevent overlap
plt.tight_layout()

# --- 4. Saving ---

os.makedirs("plots", exist_ok=True)
save_path = "plots/Dist_qc_1st_sample_in_cohort_qsmooth_no_bad_batches_shifted.png"

# plt.savefig(save_path, format="svg", transparent=True, bbox_inches="tight")
plt.savefig(save_path, dpi=300)

plt.show()

PUB_DLBCL_GSE117556 GPL14951 GPL14951_FFPE_Unknown
SOM GPL570 GPL570_Unknown_Unknown
PUB_DLBCL_NCICCR RNASeq RNASeq_FF_PolyA
PUB_DLBCL_GOYA RNASeq RNASeq_FFPE_Exome_capture
PUB_DLBCL_GSE31312 GPL570 GPL570_FFPE_Unknown
PUB_FL_CIBERSORTX GPL570 GPL570_FF_Unknown
PUB_DLBCL_GSE10846 GPL570 GPL570_FFPE_Unknown
PUB_FL_TOBIN RNASeq RNASeq_FFPE_Exome_capture
PUB_DLBCL_GSE87371 GPL570 GPL570_FF_Unknown
PUB_FL_DAVE GPL96+97 GPL96+97_FF_Unknown
PUB_DLBCL_GSE11318 GPL570 GPL570_Unknown_Unknown
PUB_FL_HUET GPL570 GPL570_FF_Unknown
Kassandra RNASeq RNASeq_FF_PolyA
PUB_DLBCL_DLC1 RNASeq RNASeq_FF_PolyA
PUB_DLBCL_GSE22898 GPL6244 GPL6244_FF_Unknown
PUB_DLBCL_JIANGGSE147986 RNASeq RNASeq_FF_PolyA
COLL_DLBCL_UMLOSSOSTRANSL RNASeq RNASeq_FFPE_Exome_capture
PUB_FL_HORN GPL96 GPL96_FFPE_Unknown
PUB_FL_BRODTKORB GPL570 GPL570_Unknown_Unknown
PUB_DLBCL_GSE34171 GPL96+97 GPL96+97_FF_Unknown
PUB_DLBCL_LYMPHOMICS RNASeq RNASeq_FF_PolyA
PUB_FL_ORICCHIO GPL96 GPL96_FF_Unknown
PUB_DLBCL_GSE23501 GPL570 GPL570_Unknown_Unknown
PUB_FL_GUO GPL570 GPL570_FF_Unknown
PUB_DLBCL_GSE19246 GPL570 GPL570_FF_Unknown
PUB_DLBCL_BAGS GPL570 GPL570_Unknown_Unknown
PUB_DLBCL_GSE93984 GPL570 GPL570_FFPE_Unknown
PUB_DLBCL_ETABM346 GPL96 GPL96_FF_Unknown
PUB_Health_B_cells_GSE154583 RNASeq RNASeq_FF_rRNADepletion
GSE85543 GPL13158 GPL13158_FF_Unknown
PUB_FL_LEBRUN GPL1708 GPL1708_FF_Unknown
PUB_DLBCL_GSE64555 GPL570 GPL570_FFPE_Unknown
GSE36907 GPL570 GPL570_FF_Unknown
MDA_Strati_FL RNASeq RNASeq_FFPE_Exome_capture
GSE56314 GPL570 GPL570_FF_Unknown
PUB_DLBCL_SHISRP312586 RNASeq RNASeq_FF_PolyA
PUB_FL_GSE62241 RNASeq RNASeq_FF_rRNADepletion
GSE107683 GPL570 GPL570_FF_Unknown
CRON3 RNASeq RNASeq_FFPE_Exome_capture
PUB_DLBCL_CARTFARAMAND RNASeq RNASeq_FF_Total
GSE12453 GPL570 GPL570_FF_Unknown
PUB_FL_NCIFFPEREBUILT RNASeq RNASeq_FFPE_Exome_capture
GSE156751 GPL17077 GPL17077_FF_Unknown
GSE89282 GPL16686 GPL16686_FF_Unknown
PUB_DLBCL_NCIACALA RNASeq RNASeq_FFPE_PolyA
GSE22886 GPL96+97 GPL96+97_FF_Unknown
GSE12195 GPL570 GPL570_FF_Unknown
PUB_Health_B_cells_GSE119234 RNASeq RNASeq_FF_rRNADepletion
GSE64028 GPL570 GPL570_FF_Unknown
PUB_FL_NCISTAUDT RNASeq RNASeq_FF_Total
PUB_Health_B_cells_GSE111310 RNASeq RNASeq_FF_PolyA
PUB_Health_B_cells_GSE139833 RNASeq RNASeq_FF_Unknown
GSE141005 GPL16686 GPL16686_FF_Unknown
Leandro-Normals RNASeq RNASeq_FF_Unknown
GSE17186 GPL570 GPL570_FF_Unknown
GSE12845 GPL96+97 GPL96+97_FF_Unknown
GSE12366 GPL570 GPL570_FF_Unknown
PUB_FL_PARSA RNASeq RNASeq_FFPE_PolyA
GSE136248 GPL570 GPL570_FF_Unknown
PUB_Health_B_cells_GSE92387 RNASeq RNASeq_FF_PolyA
GSE136249 GPL570 GPL570_FF_Unknown
GSE51528 GPL6244 GPL6244_FF_Unknown
PUB_Suntsova_GSE120795 GPL570 GPL570_FF_Unknown
PUB_DLBCL_PARKSRP100394 RNASeq RNASeq_FF_PolyA
GSE144089 GPL17077 GPL17077_FF_Unknown
PUB_Health_B_cells_GSE110669 RNASeq RNASeq_FF_PolyA
PUB_DLBCL_YANCART RNASeq RNASeq_FF_PolyA
GSE15271 GPL570 GPL570_FF_Unknown
PUB_Beguelin_GSE45982 RNASeq RNASeq_FF_PolyA
PUB_Health_B_cells_GSE115655 RNASeq RNASeq_FF_PolyA
GSE13411 GPL96 GPL96_FF_Unknown
GSE38697 GPL570 GPL570_FF_Unknown
E-MTAB-3974 GPL16686 GPL16686_FF_Unknown
GSE55267 GPL570 GPL570_FF_Unknown
GSE19599 GPL570 GPL570_FF_Unknown
GSE61912 GPL10739 GPL10739_FF_Unknown
PUB_Health_Bunting_GSE84022 RNASeq RNASeq_FF_Unknown
PUB_Pastore_GSE119103 RNASeq RNASeq_FF_PolyA

In [ ]:
cohort = "Kassandra"  #'PUB_DLBCL_GSE19246'

exp_local = comb_exp.T[comb_ann.index[comb_ann.Cohort_general == cohort]].dropna()
ann_local = comb_ann.loc[comb_ann.Cohort_general == cohort,]

platform = comb_ann.loc[comb_ann.Cohort_general == cohort, "PLATFORM_RNA"][0]
batch = comb_ann.loc[comb_ann.Cohort_general == cohort, "RNA_BATCH"][0]

In [ ]:
sns.histplot(
    data=np.log2(exp_local + 1),
    x=exp_local.columns[70],
    bins=50,
    # color="#555555",
    # edgecolor='white',
    linewidth=0.1,
)

Training Distribution Matching (TDM)Training Distribution Matching (TDM) is an algorithm specifically developed to address "dataset shift" when combining microarray and RNA-seq data. The core objective of TDM is to transform a "test" distribution to match the "training" distribution of a reference dataset while preserving the internal rank-order relationships between samples.To convert a microarray cohort into a bimodal RNA-seq format, one would use a bimodal RNA-seq dataset (such as the Suntsova atlas, GSE120795) as the target training distribution. 1  TDM establishes a mapping between the quantiles of the array data and the quantiles of the target seq data. If the target distribution has a significant zero mass, the TDM transformation will compress the lower intensities of the microarray into a zero or near-zero peak, thereby re-creating the bimodal profile. 2 

#### Feature specific quantile and mean-variance normalization

In [ ]:
def fsqn_normalize(data, groups, target_group=None):
    """
    Implementation of Feature Specific Quantile Normalization (FSQN).
    FSQN transforms the distribution of each feature (gene) within each group
    to match a target distribution (either a specific group or the global mean).

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    groups : pd.Series or list
        The grouping variable (e.g., RNA_BATCH).
    target_group : str, optional
        The name of the group to use as the reference distribution.
        If None, the global average distribution across all samples is used.

    Returns:
    --------
    pd.DataFrame
        FSQN-normalized expression data.
    """
    # Align groups with data index
    if not isinstance(groups, pd.Series):
        groups = pd.Series(groups, index=data.index)
    groups = groups.loc[data.index]

    unique_groups = np.unique(groups)
    normalized_data = data.copy()

    # 1. Define the target distribution
    # FSQN usually targets the distribution of a 'reference' batch or the global mean
    if target_group and target_group in unique_groups:
        target_dist = np.sort(data.loc[groups == target_group].values, axis=1).mean(
            axis=0
        )
    else:
        # Use global mean distribution as target
        target_dist = np.sort(data.values, axis=1).mean(axis=0)

    n_genes = data.shape[1]
    # Create an interpolation line for the target distribution
    target_ranks = np.linspace(0, 1, n_genes)

    # 2. Normalize each group to the target
    for g in unique_groups:
        group_indices = data.index[groups == g]
        group_data = data.loc[group_indices].values

        for i in range(len(group_indices)):
            sample_values = group_data[i, :]
            # Get the rank order of values in the current sample
            order = np.argsort(sample_values)

            # Map the target distribution values onto the sample's ranks
            # This forces the sample to have the target distribution
            normalized_data.loc[group_indices[i]].iloc[order] = target_dist

    return normalized_data


def fsmvn_normalize(data, groups):
    """
    Implementation of Feature-Specific Mean–Variance Normalization (FSMVN).
    This method scales each feature (gene) within each group so that they all
    share a common mean and variance, effectively removing batch-specific
    scaling issues while preserving feature-specific information.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    groups : pd.Series or list
        The grouping variable (e.g., RNA_BATCH).

    Returns:
    --------
    pd.DataFrame
        FSMVN-normalized expression data.
    """
    # Align groups with data index
    if not isinstance(groups, pd.Series):
        groups = pd.Series(groups, index=data.index)
    groups = groups.loc[data.index]

    unique_groups = np.unique(groups)
    normalized_data = data.copy()

    # 1. Calculate global target stats (grand mean and grand std per gene)
    global_mean = data.mean(axis=0)
    global_std = data.std(axis=0).replace(0, 1)  # Prevent division by zero

    # 2. Adjust each group's features to match global targets
    for g in unique_groups:
        group_mask = groups == g
        group_data = data.loc[group_mask]

        # Calculate group-specific stats per gene
        group_mean = group_data.mean(axis=0)
        group_std = group_data.std(axis=0).replace(0, 1)

        # Z-score the group data and then re-scale to global parameters
        # formula: x_norm = ((x - group_mean) / group_std) * global_std + global_mean
        z_scored = (group_data - group_mean) / group_std
        rescaled = (z_scored * global_std) + global_mean

        normalized_data.loc[group_mask] = rescaled

    return normalized_data


# Example Usage:
# --------------
# normalized_fsqn = fsqn_normalize(comp_exp, comb_ann['RNA_BATCH'])
# normalized_fsmvn = fsmvn_normalize(comp_exp, comb_ann['RNA_BATCH'])

#### !!! Full script to get comparison table against SOM !!!

In [ ]:
def apply_fsqn_r(data, annotation, batch_col='RNA_BATCH', target_group=None):
    """
    Applies Feature Specific Quantile Normalization (FSQN) using the original R package 
    (https://github.com/jenniferfranks/FSQN) via rpy2.
    
    This function normalizes the feature (gene) distributions of each batch to match 
    the feature distributions of a specified target reference group.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations containing the batch groupings.
    batch_col : str
        Column containing the batch variable. 
    target_group : str
        The name of the group in `batch_col` to use as the reference distribution.
        If None, the global dataset is used as the reference (though specifying a 
        high-quality batch like 'RNASeq_FF_PolyA' is recommended).
        
    Returns:
    --------
    pd.DataFrame
        FSQN-corrected expression data (Samples as rows, Genes as columns).
    """
    print("Loading R packages for FSQN...")
    
    base = importr('base')
    
    # Attempt to load the FSQN package
    try:
        fsqn_pkg = importr('FSQN')
    except Exception as e:
        raise ImportError(
            "The FSQN R package is not installed in your R environment. "
            "Please open R and run:\n"
            "  install.packages('devtools')\n"
            "  devtools::install_github('jenniferfranks/FSQN')"
        ) from e
    
    # 1. Align data and annotation (FSQN expects Samples as rows, Genes as columns)
    common_samples = data.index.intersection(annotation.index)
    X = data.loc[common_samples].fillna(0).astype(float)
    meta = annotation.loc[common_samples]
    
    # 2. Extract the Reference Matrix
    if target_group and target_group in meta[batch_col].values:
        print(f"Using '{target_group}' as the reference distribution.")
        ref_samples = meta[meta[batch_col] == target_group].index
        ref_df = X.loc[ref_samples]
    else:
        print("No valid target_group specified. Using the entire dataset as the reference distribution.")
        ref_df = X
        
    # 3. Iterate through batches and apply FSQN using a robust local converter
    unique_batches = meta[batch_col].unique()
    normalized_dfs = []
    
    # localconverter securely switches Python-R conversions to handle Numpy Arrays seamlessly
    with localconverter(ro.default_converter + numpy2ri.converter):
        # Convert Reference Matrix to R numeric matrix
        r_ref_mat = ro.conversion.py2rpy(ref_df.values)
        
        for batch in unique_batches:
            print(f"Normalizing batch: {batch}...")
            batch_samples = meta[meta[batch_col] == batch].index
            batch_df = X.loc[batch_samples]
            
            # If the batch is exactly the target group, we can just append it directly
            # to avoid unnecessarily quantile normalizing a matrix against itself
            if batch == target_group:
                normalized_dfs.append(batch_df)
                continue
                
            # Convert Batch Matrix to R numeric matrix
            r_batch_mat = ro.conversion.py2rpy(batch_df.values)
            
            # Run FSQN
            # The main function in the FSQN package is `quantileNormalizeByFeature`
            try:
                if hasattr(fsqn_pkg, 'quantileNormalizeByFeature'):
                    r_norm_mat = fsqn_pkg.quantileNormalizeByFeature(r_batch_mat, r_ref_mat)
                else:
                    # Robust fallback: evaluate directly in R if rpy2 namespace mapping missed it
                    ro.r('library(FSQN)')
                    ro.globalenv['test_mat'] = r_batch_mat
                    ro.globalenv['ref_mat'] = r_ref_mat
                    r_norm_mat = ro.r('quantileNormalizeByFeature(test_mat, ref_mat)')
                
                # Catch R returning NULL safely avoiding NumPy element-wise comparisons
                if type(r_norm_mat).__name__ == 'NULLType' or r_norm_mat is ro.NULL:
                    raise RuntimeError(
                        "FSQN returned NULL in R. This usually means the matrices have a different number of columns (genes)."
                    )
                    
            except Exception as e:
                raise RuntimeError(f"Error executing FSQN in R: {str(e)}")
            
            # Extract back to NumPy
            norm_matrix = ro.conversion.rpy2py(r_norm_mat)
            
            # Create DataFrame (Samples x Genes)
            norm_df = pd.DataFrame(
                norm_matrix, 
                index=batch_df.index, 
                columns=batch_df.columns
            )
            
            normalized_dfs.append(norm_df)
            
    # 4. Recombine all normalized batches
    print("Recombining normalized batches...")
    final_normalized_data = pd.concat(normalized_dfs, axis=0)
    
    # Ensure the original sample order is maintained
    final_normalized_data = final_normalized_data.loc[common_samples]
    
    print("FSQN complete.")
    return final_normalized_data


# ==========================================
# EXAMPLE USAGE
# ==========================================
# fsqn_normalized_exp = apply_fsqn_r(
#     comp_exp, 
#     comb_ann, 
#     batch_col='RNA_BATCH', 
#     target_group='RNASeq_FF_PolyA'  # Choose your highest quality/largest batch here
# )

comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann = pd.read_csv(
    "comb_ann_unified.csv", sep=","
).set_index("Unnamed: 0")


bad_batches = ["GPL14951_FFPE_Unknown", "GPL8432_FFPE_Unknown", "GPL13938_FFPE_Unknown"]
bad_cohorts = ["SOM"]


# "GPL8432_FFPE_Unknown",
# "GPL6244_FFPE_Unknown",
# "GPL17586_FF_Unknown",
# "GPL20188_FF_Unknown",
# "GPL23541_FF_Unknown",
# "GPL887_FFPE_Unknown",
# "GPL26356_FF_Unknown",
# "GPL17047_FF_Unknown",
# "GPL6244_FF_Unknown",
# "GPL13938_FFPE_Unknown",
# ]
rare_groups = [
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
    "Other",
]


#comb_ann = comb_ann[~comb_ann.RNA_BATCH.isin(bad_batches)]
comb_ann = comb_ann[~comb_ann.Major_group.isin(rare_groups)]
#comb_ann = comb_ann[~comb_ann.COHORT_LABEL.isin(bad_cohorts)]

#comb_ann.loc[
#    comb_ann.Major_group.isin(["Normal_B_cells", "Kassandra"]),
#    ["Major_group"],
#] = "Normal_B_cells"

comb_ann['Diagnosis_with_coo'] = comb_ann.Diagnosis_cell_type_general + '_' + comb_ann['coo_bg'].fillna('')
comb_ann['Diagnosis_unified_with_coo'] = comb_ann.Diagnosis_cell_type_unified + '_' + comb_ann['coo_bg'].fillna('')



#comb_ann.loc[
#    comb_ann.Major_group.isin(["Normal_B_cells", "Kassandra"]),
#    ["Major_group"],
#] = "Normal_B_cells"



common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
filtered_exp = comb_exp.loc[common_samples].dropna(axis=1)
filtered_exp = filtered_exp.loc[~filtered_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
batch_groups = comb_ann.loc[common_samples, "RNA_BATCH"]
batch_groups = batch_groups.loc[~batch_groups.index.duplicated(keep="first"),]
comb_ann_dedup = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]
fsqn_normalized_exp = apply_fsqn_r(
    filtered_exp, 
    comb_ann_dedup, 
    batch_col='RNA_BATCH', 
    target_group='RNASeq_FF_PolyA'  # Choose your highest quality/largest batch here
)

In [ ]:
fsqn_normalized_exp = apply_fsqn_r(
    filtered_exp, 
    comb_ann_dedup, 
    batch_col='RNA_BATCH', 
    target_group='RNASeq_FF_PolyA'  # Choose your highest quality/largest batch here
)

In [ ]:
fsqn_normalized_exp.to_csv(
    os.path.join(FL_DATA_ROOT, "fsqn_normalized_exp_R.tsv"), sep="\t"
)

In [ ]:
fsqn_normalized_exp

In [ ]:
normalized_fsmvn = fsmvn_normalize(filtered_exp, comb_ann_dedup["RNA_BATCH"])

In [ ]:
normalized_fsqn = fsqn_normalize(filtered_exp, comb_ann_dedup["RNA_BATCH"])

In [ ]:
normalized_fsqn

In [ ]:
cohorts_counts = comb_ann_dedup.Cohort_general.fillna("na").value_counts()

GLOBAL_FONT_SIZE = 7.5
FIGURE_SIZE = (30, 20)
BINS = 30

# --- 3. Visualization ---

figure, axs = plt.subplots(
    nrows=8,
    ncols=11,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=FIGURE_SIZE,
)

# Flatten axes for iteration
flat_axs = axs.flatten()

for cohort, ax in zip(cohorts_counts.index, flat_axs):

    # Filter the data
    exp_local = normalized_fsmvn.T[
        comb_ann_dedup.index[comb_ann_dedup.Cohort_general == cohort]
    ].dropna()

    platform = comb_ann_dedup.loc[
        comb_ann_dedup.Cohort_general == cohort, "PLATFORM_RNA"
    ][0]
    batch = comb_ann_dedup.loc[comb_ann_dedup.Cohort_general == cohort, "RNA_BATCH"][0]
    print(cohort, platform, batch)
    if batch == "GPL6244_FFPE_Unknown":
        print(comb_ann_dedup[comb_ann_dedup.RNA_BATCH == batch])

    if exp_local.max().max() > 50:
        exp_local = np.log2(exp_local + 1)

    # Plot histogram
    sns.histplot(
        data=exp_local,
        x=exp_local.columns[0],
        ax=ax,
        bins=BINS,
        # color="#555555",
        # edgecolor='white',
        linewidth=0.1,
    )

    # --- Unified Font Size Customization ---

    # Subplot Title
    ax.set_title(
        f"{cohort} \n {platform} {batch}", fontsize=GLOBAL_FONT_SIZE, fontweight="bold"
    )

    # Axis Labels
    ax.set_xlabel("Gene expression value", fontsize=GLOBAL_FONT_SIZE)
    ax.set_ylabel("Frequency", fontsize=GLOBAL_FONT_SIZE)

    # Tick Labels (X and Y)
    ax.tick_params(axis="both", which="major", labelsize=GLOBAL_FONT_SIZE)

# Adjust layout to prevent overlap
plt.tight_layout()

# --- 4. Saving ---

os.makedirs("plots", exist_ok=True)
save_path = "plots/Dist_qc_1st_sample_in_cohort_fsmvn.png"

# plt.savefig(save_path, format="svg", transparent=True, bbox_inches="tight")
plt.savefig(save_path, dpi=300)

plt.show()

In [ ]:
normalized_fsqn.to_csv(
    "$FL_DATA_ROOT/exp_normalized_fsqn.tsv", sep="\t"
)

In [ ]:
comb_ann_dedup.to_csv(
    os.path.join(FL_DATA_ROOT, "ann_normalized_fsqn_R.tsv"), sep="\t"
)

In [ ]:
normalized_fsqn

#### Harmony-like normalization

In [ ]:
def harmony_normalize(
    data,
    annotation,
    batch_col="RNA_BATCH",
    bio_col="Major_group",
    n_components=50,
    max_iter=10,
    sigma=0.1,
):
    """
    Implementation of the Harmony algorithm adapted for bulk gene expression data.
    Reference: Korsunsky et al., Nature Methods 2019 (https://doi.org/10.1038/s41592-019-0619-0)

    This function performs batch correction in PCA space and then projects the corrected
    results back into the original gene expression space.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations containing batch and biological metadata.
    batch_col : str
        Column in annotation representing technical batches to be removed.
    bio_col : str
        Column in annotation representing biological groups to be preserved.
    n_components : int
        Number of PCA components to use for the integration space.
    max_iter : int
        Maximum number of Harmony iterations.
    sigma : float
        Soft clustering parameter. Lower values make clustering "harder".

    Returns:
    --------
    pd.DataFrame
        The integrated gene expression data (normalized genes as columns).
    """
    # 1. Align data and annotation
    common_idx = data.index.intersection(annotation.index)
    X_raw = data.loc[common_idx].fillna(0)
    X = X_raw.values
    meta = annotation.loc[common_idx]

    # 2. Standardize and Initial PCA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    Y = pca.fit_transform(X_scaled).T  # Y is [d x N] where d is PCs and N is samples

    # 3. Setup Harmony variables
    N = Y.shape[1]
    phi = pd.get_dummies(meta[batch_col]).values.T  # Batch design matrix [B x N]
    n_batch = phi.shape[0]

    # Initialize probabilities R [K x N]
    K = len(meta[bio_col].unique()) * 2
    R = np.ones((K, N)) / K
    O = np.zeros((Y.shape[0], K))

    # 4. Harmony Iterative Integration
    for i in range(max_iter):
        if i == 0:
            from sklearn.cluster import KMeans

            kmeans = KMeans(n_clusters=K, n_init=1).fit(Y.T)
            O = kmeans.cluster_centers_.T

        # Calculate soft assignment R
        dist = np.dot(O.T, Y)
        # Stability trick for exponential
        dist_max = np.max(dist, axis=0)
        R = np.exp((dist - dist_max) / sigma)
        R /= np.sum(R, axis=0)

        # Update centroids O based on R
        for k in range(K):
            if np.sum(R[k, :]) > 0:
                O[:, k] = np.average(Y, axis=1, weights=R[k, :])

        # Integrate (Maximization step)
        for k in range(K):
            r_k = R[k, :]
            for b in range(n_batch):
                idx = phi[b, :] == 1
                if not np.any(idx):
                    continue

                weights = r_k[idx]
                if np.sum(weights) == 0:
                    continue

                batch_mean = np.average(Y[:, idx], axis=1, weights=weights)
                global_mean = np.average(Y, axis=1, weights=r_k)

                # Correct the samples in this batch by moving them toward the global mean
                Y[:, idx] -= (batch_mean - global_mean)[:, np.newaxis]

    # 5. Project back to Gene Space
    # Y is corrected PCA [d x N]. Transpose to [N x d]
    Y_corrected = Y.T

    # Inverse transform: (PCs * Loadings) + Mean
    X_corrected_scaled = np.dot(Y_corrected, pca.components_)

    # Reverse the StandardScaler
    X_final = (X_corrected_scaled * scaler.scale_) + scaler.mean_

    # Return as DataFrame with original gene names
    normalized_df = pd.DataFrame(X_final, index=common_idx, columns=X_raw.columns)

    return normalized_df


# Example Usage:
# --------------
# integrated_pca = harmony_normalize(
#     comp_exp,
#     comb_ann,
#     batch_col='RNA_BATCH',
#     bio_col='Diagnosis_cell_type_general'
# )

In [ ]:
harmony_normalized_exp = harmony_normalize(
    filtered_exp, comb_ann_dedup, batch_col="RNA_BATCH", bio_col="Major_group"
)

In [ ]:
print((~harmony_normalized_exp.isna()).sum().sort_values().to_string())

In [ ]:
exp_local[exp_local.columns[0]].quantile(0.01)

In [ ]:
exp_local[exp_local.columns[0]].quantile(0.99)

In [ ]:
cohorts_counts = comb_ann_dedup.Cohort_general.fillna("na").value_counts()

GLOBAL_FONT_SIZE = 7.5
FIGURE_SIZE = (30, 20)
BINS = 30

# --- 3. Visualization ---

figure, axs = plt.subplots(
    nrows=8,
    ncols=11,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=FIGURE_SIZE,
)

# Flatten axes for iteration
flat_axs = axs.flatten()

for cohort, ax in zip(cohorts_counts.index, flat_axs):

    # Filter the data
    exp_local = harmony_normalized_exp.T[
        comb_ann_dedup.index[comb_ann_dedup.Cohort_general == cohort]
    ].dropna()

    platform = comb_ann_dedup.loc[
        comb_ann_dedup.Cohort_general == cohort, "PLATFORM_RNA"
    ][0]
    batch = comb_ann_dedup.loc[comb_ann_dedup.Cohort_general == cohort, "RNA_BATCH"][0]
    print(cohort, platform, batch)
    exp_local = exp_local.clip(
        exp_local[exp_local.columns[0]].quantile(0.01),
        exp_local[exp_local.columns[0]].quantile(0.99),
    )

    #    if exp_local.max().max() > 50:
    #        exp_local = np.log2(exp_local + 1)

    # Plot histogram

    sns.histplot(
        data=exp_local,
        x=exp_local.columns[0],
        ax=ax,
        bins=BINS,
        # color="#555555",
        # edgecolor='white',
        linewidth=0.1,
    )

    # --- Unified Font Size Customization ---

    # Subplot Title
    ax.set_title(
        f"{cohort} \n {platform} {batch}", fontsize=GLOBAL_FONT_SIZE, fontweight="bold"
    )

    # Axis Labels
    ax.set_xlabel("Gene expression value", fontsize=GLOBAL_FONT_SIZE)
    ax.set_ylabel("Frequency", fontsize=GLOBAL_FONT_SIZE)

    # Tick Labels (X and Y)
    ax.tick_params(axis="both", which="major", labelsize=GLOBAL_FONT_SIZE)

# Adjust layout to prevent overlap
plt.tight_layout()

# --- 4. Saving ---

os.makedirs("plots", exist_ok=True)
save_path = "plots/Dist_qc_1st_sample_in_cohort_harmony.png"

# plt.savefig(save_path, format="svg", transparent=True, bbox_inches="tight")
plt.savefig(save_path, dpi=300)

plt.show()

#### ComBat - Empirical Bayes

In [ ]:
# Attempt to import the Python port of ComBat
try:
    from neuroCombat import neuroCombat
except ImportError:
    warnings.warn(
        "neuroCombat is not installed. To use the native Python implementation, please run: `pip install neuroCombat`"
    )

# Attempt to import rpy2 for the Bioconductor sva::ComBat approach
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
except ImportError:
    pass


def apply_combat_neuro(data, annotation, batch_col="RNA_BATCH", bio_col="Major_group"):
    """
    Python-native implementation of ComBat Empirical Bayes normalization using neuroCombat.
    It removes technical batch effects while preserving known biological covariates.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations.
    batch_col : str
        Column containing the batch variable to correct (e.g., 'RNA_BATCH').
    bio_col : str
        Column containing the biological group to preserve (e.g., 'Major_group').

    Returns:
    --------
    pd.DataFrame
        ComBat-corrected expression data (Samples as rows).
    """
    print(f"Preparing data for native Python neuroCombat...")

    # 1. Align data and annotation
    common_samples = data.index.intersection(annotation.index)
    X_raw = data.loc[common_samples]
    meta = annotation.loc[common_samples]

    # ComBat requires complete cases (no NaNs). Fill missing with 0.
    X_filled = X_raw.fillna(0).astype(float)

    # neuroCombat expects data in shape: Features (Genes) x Samples
    X = X_filled.T

    # Prepare covariate DataFrame
    covars = meta[[batch_col, bio_col]].copy()

    # 2. Run neuroCombat
    print(
        f"Applying ComBat normalization...\nCorrecting batch: '{batch_col}'\nPreserving biology: '{bio_col}'"
    )

    combat_results = neuroCombat(
        dat=X, covars=covars, batch_col=batch_col, categorical_cols=[bio_col]
    )

    # 3. Transpose back to (Samples x Genes) and restore original names
    corrected_matrix = combat_results["data"]
    corrected_df = pd.DataFrame(
        corrected_matrix.T, index=common_samples, columns=X_raw.columns
    )

    return corrected_df


def apply_combat_sva(data, annotation, batch_col="RNA_BATCH", bio_col="Major_group"):
    """
    Bioconductor implementation of ComBat using rpy2 (sva::ComBat).
    This acts as a robust fallback to the gold-standard R implementation if
    native Python library versioning issues arise.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations.
    batch_col : str
        Column containing the batch variable to correct.
    bio_col : str
        Column containing the biological group to preserve.

    Returns:
    --------
    pd.DataFrame
        ComBat-corrected expression data (Samples as rows).
    """
    print(f"Loading R package 'sva' for ComBat batch correction...")
    pandas2ri.activate()

    sva = importr("sva")
    stats = importr("stats")
    base = importr("base")

    # Align and transpose (R expects Genes x Samples)
    common_samples = data.index.intersection(annotation.index)
    X = data.loc[common_samples].fillna(0).astype(float).T

    # Convert to R Matrix
    r_X_mat = base.as_matrix(pandas2ri.py2rpy(X))

    # Define Batch Covariate
    batches = annotation.loc[common_samples, batch_col].astype(str).values
    r_batches = ro.StrVector(batches)

    # Define Biological Covariate (To preserve)
    bio_groups = annotation.loc[common_samples, bio_col].astype(str).values
    ro.globalenv["bio_groups"] = ro.StrVector(bio_groups)

    # Create the model matrix for the biological covariate
    # ComBat requires the mod matrix to NOT include the batch variable itself
    mod = stats.model_matrix(ro.Formula("~ as.factor(bio_groups)"))

    # Run sva::ComBat
    print(
        f"Applying sva::ComBat...\nCorrecting batch: '{batch_col}'\nPreserving biology: '{bio_col}'"
    )
    corrected_r = sva.ComBat(dat=r_X_mat, batch=r_batches, mod=mod, par_prior=True)

    # Extract back to Pandas (Checking if rpy2 auto-converted to numpy)
    if isinstance(corrected_r, np.ndarray):
        corrected_matrix = corrected_r
    else:
        corrected_matrix = pandas2ri.rpy2py(corrected_r)

    corrected_df = pd.DataFrame(corrected_matrix, index=X.index, columns=X.columns).T

    return corrected_df


# ==========================================
# EXAMPLE USAGE
# ==========================================
# Option A: Pure Python Implementation (Requires `pip install neuroCombat`)
# combat_normalized_exp = apply_combat_neuro(
#     comp_exp,
#     comb_ann,
#     batch_col='RNA_BATCH',
#     bio_col='Major_group'
# )
#
# Option B: R/Bioconductor Implementation (Requires `rpy2` and Bioconductor `sva`)
# combat_normalized_exp_r = apply_combat_sva(
#     comp_exp,
#     comb_ann,
#     batch_col='RNA_BATCH',
#     bio_col='Major_group'
# )

In [ ]:
combat_normalized_exp = apply_combat_neuro(
     filtered_exp,
     comb_ann_dedup,
     batch_col='RNA_BATCH',
     bio_col='Major_group'
)

In [ ]:
combat_normalized_exp

#### Quantile normalization

In [ ]:
def quantile_normalize(data):
    """
    Implements standard Quantile Normalization for bulk gene expression data.
    Forces all samples in the dataset to have the exact same statistical distribution.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
        
    Returns:
    --------
    pd.DataFrame
        Quantile-normalized expression data.
    """
    print(f"Applying standard Quantile Normalization to {data.shape[0]} samples and {data.shape[1]} genes...")
    
    # 1. Transpose the data
    # It is computationally easier to rank and sort down columns.
    # We transpose so Samples = columns, Genes = rows.
    data_t = data.T
    
    # 2. Sort each sample independently
    # This gives us the ordered expression values for every sample
    sorted_data = np.sort(data_t.values, axis=0)
    
    # 3. Calculate the Target Distribution
    # The target is the mean of each rank across all samples
    target_dist = np.mean(sorted_data, axis=1)
    
    # 4. Rank the original data
    # We use method='average' to properly handle identical expression values (ties).
    # If a sample has 100 genes with expression 0.0, they share the average fractional rank.
    # We subtract 1 to convert pandas 1-based ranking to 0-based indexing for numpy.
    ranks = data_t.rank(method='average').values - 1
    
    # 5. Map the target distribution back to the samples
    norm_matrix = np.empty_like(data_t.values)
    
    # We loop through each sample and map its ranks to the target distribution.
    # np.interp safely maps fractional ranks (from ties) to the exact interpolated target value.
    for i in range(data_t.shape[1]):
        norm_matrix[:, i] = np.interp(
            ranks[:, i], 
            np.arange(len(target_dist)), 
            target_dist
        )
        
    # 6. Transpose back to original shape and attach metadata
    normalized_df = pd.DataFrame(
        norm_matrix, 
        index=data_t.index, 
        columns=data_t.columns
    ).T
    
    print("Quantile Normalization complete.")
    return normalized_df


In [ ]:
normalized_quantile = quantile_normalize(filtered_exp)

In [ ]:
normalized_quantile

In [ ]:
sns.distplot()

#### PCA  & UMAP & TSNE before and after normalization

In [ ]:
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann

In [ ]:
Platform_general                                                                            6577
Diagnosis_cell_type                                                                         6593
Cohort_general                                                                              6604
TUMOR_NORMAL                                                                                6604
SAMPLE_NAME                                                                                 6604
RNA_BATCH                                                                                   6604
COHORT_LABEL                                                                                6604
Diagnosis_cell_type_general                                                                 6604
PLATFORM_RNA                                                                                6604
RNASEQ_SOURCE                                                                               6604
Major_group                                                                                 6604

In [ ]:
comb_ann_dedup.Platform_general.unique()

In [ ]:
fsqn_normalized_exp.dropna()

In [ ]:
column = "Diagnosis_unified_with_coo"
palette = lymphoma_ontogeny_palette#cohort_palette #rna_batch_palette  # cohort_palette #rna_batch_palette # # #platform_palette #
# filtering = comb_ann_dedup.RNA_BATCH.isin(['GPL570_Unknown_Unknown', 'RNASeq_FFPE_Exome_capture'])
filtering = ~comb_ann_dedup.RNA_BATCH.isna()

figure, axs = plt.subplots(
    nrows=1, ncols=2, sharex=False, sharey=False, squeeze=True, figsize=(15, 7.5)
)

pca_plot(
    filtered_exp[filtering],
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="Non-normalized",
    palette=palette,
)

pca_plot(
    np.log2(normalized_quantile[filtering] + 1),  # normalized_exp, #normalized_fsqn
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="Normalized",
    palette=palette,
)

In [ ]:
column = "Diagnosis_unified_with_coo"
palette = lymphoma_ontogeny_palette#cohort_palette #rna_batch_palette  # cohort_palette #rna_batch_palette # # #platform_palette #

figure, axs = plt.subplots(
    nrows=1,
    ncols=2,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=(10, 5),
)

umap_plot(
    filtered_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="Non-normalized",
    palette=palette,
)

umap_plot(
    np.log2(normalized_quantile + 1),  # normalized_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="Normalized",
    palette=palette,
)

In [ ]:
column = "Diagnosis_unified_with_coo"
palette = lymphoma_ontogeny_palette#cohort_palette #rna_batch_palette  # cohort_palette #rna_batch_palette # # #platform_palette #

figure, axs = plt.subplots(
    nrows=1,
    ncols=2,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=(10, 5),
)

tsne_plot(
    filtered_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="Non-normalized",
    palette=palette,
)

tsne_plot(
    np.log2(normalized_quantile + 1),  # normalized_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="Normalized",
    palette=palette,
)



In [ ]:
comb_ann_dedup.PLATFORM_RNA.value_counts()

In [ ]:
pd.crosstab(comb_ann_dedup.RNA_BATCH, comb_ann_dedup.Major_group)

In [ ]:
pca_plot(
    normalized_exp,
    grouping=comb_ann_dedup["Platform_classified"].fillna("na"),
    legend="out",
)

## Trial batch correction and signatures analysis according to Sasha Nesmelov

### Dist QC for RNA by cohort 

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann

In [ ]:
comb_ann[["RNA_BATCH", "COHORT_LABEL", "RNASEQ_SOURCE", "Diagnosis_cell_type_general"]]

In [ ]:
for cohort in cohorts_counts.index:
    print(cohort)

In [ ]:
quantiles_clip = False


cohorts_counts = comb_ann.Cohort_general.fillna("na").value_counts()

GLOBAL_FONT_SIZE = 7.5
FIGURE_SIZE = (30, 30)
BINS = 30

# --- 3. Visualization ---

figure, axs = plt.subplots(
    nrows=10,
    ncols=9,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=FIGURE_SIZE,
)

# Flatten axes for iteration
flat_axs = axs.flatten()


for cohort, ax in zip(cohorts_counts.index, flat_axs):
    # Filter the data
    exp_local = fsqn_normalized_exp.dropna().T[comb_ann.index[comb_ann.Cohort_general == cohort]].dropna()
    
    platform = comb_ann.loc[comb_ann.Cohort_general == cohort, "PLATFORM_RNA"][0]
    batch = comb_ann.loc[comb_ann.Cohort_general == cohort, "RNA_BATCH"][0]
    if exp_local.max().max() > 50:
        exp_local = np.log2(exp_local + 2)
    if quantiles_clip:
        top_quantile = exp_local[exp_local.columns[0]].quantile(0.99)
        bottom_quantile = exp_local[exp_local.columns[0]].quantile(0.01)
        exp_local = exp_local.clip(bottom_quantile, top_quantile)
    # Plot histogram
    sns.histplot(
        data=exp_local,
        x=exp_local.columns[0],
        ax=ax,
        bins=BINS,
        # color="#555555",
        # edgecolor='white',
        linewidth=0.1,
    )

    # --- Unified Font Size Customization ---

    # Subplot Title
    ax.set_title(
        f"{cohort} \n {platform} {batch}", fontsize=GLOBAL_FONT_SIZE, fontweight="bold"
    )

    # Axis Labels
    ax.set_xlabel("Gene expression value", fontsize=GLOBAL_FONT_SIZE)
    ax.set_ylabel("Frequency", fontsize=GLOBAL_FONT_SIZE)

    # Tick Labels (X and Y)
    ax.tick_params(axis="both", which="major", labelsize=GLOBAL_FONT_SIZE)

# Adjust layout to prevent overlap
plt.tight_layout()

# --- 4. Saving ---

os.makedirs("plots", exist_ok=True)
save_path = "plots/Dist_qc_1st_sample_in_cohort_FSQN_R.png"

# plt.savefig(save_path, format="svg", transparent=True, bbox_inches="tight")
plt.savefig(save_path, dpi=300)

plt.show()

In [ ]:
for cohort in cohorts_counts.index:
    exp_local = comb_exp[comb_ann.index[comb_ann.Cohort_general == cohort]]

In [ ]:
exp_local.columns[0]

### Additional annotation preparation

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann

In [ ]:
(~comb_exp.isna()).sum().sort_values().reset_index().plot()

In [ ]:
comb_ann.Major_group.value_counts()

In [ ]:
comb_ann["Diagnosis_cell_type"] = comb_ann.DIAGNOSIS
comb_ann["Diagnosis_cell_type"][comb_ann["Diagnosis_cell_type"].isna()] = comb_ann[
    "Cell_type"
][comb_ann["Diagnosis_cell_type"].isna()]
comb_ann["Diagnosis_cell_type_general"] = comb_ann["Diagnosis_cell_type"]
comb_ann["Diagnosis_cell_type_general"][
    comb_ann.index.isin(ann_normal.index)
] = "Normal_B_cells"


comb_ann["Platform_general"] = comb_ann.PLATFORM_RNA
comb_ann["Platform_general"][comb_ann["PLATFORM_RNA"].isna()] = comb_ann["Platform"][
    comb_ann["PLATFORM_RNA"].isna()
]
comb_ann["Platform_general"][
    comb_ann.index.isin(kassandra_ann["Unnamed: 0"])
] = "Kassandra"

In [ ]:
sum(comb_ann["Platform_general"].isna())

In [ ]:
cells_ann.Cohort.fillna("na").value_counts()

In [ ]:
cells_ann = comb_ann[comb_ann.DIAGNOSIS.isna()]
print((~cells_ann.isna()).sum().sort_values().to_string())

In [ ]:
norm.Cell_type.unique()

In [ ]:
comb_ann.Platform_general.unique()

In [ ]:
comb_ann.Diagnosis_cell_type.fillna("NA").unique()

### Calculate and median scale processes for the entire dataset

In [ ]:
dlbcl_gmt = read_gene_sets(os.path.join(FL_SHARED_ROOT, "MFP_REFERENCE", "dlbcl_signatures.gmt"))
signatures = ssgsea_formula(comb_exp.T, dlbcl_gmt)
progeny = run_progeny(comb_exp.T)

In [ ]:
comb_processes_scaled = median_scale(data=signatures, axis=1)
comb_processes_scaled

In [ ]:
median_scale

In [ ]:
comb_processes_scaled.median(axis=1)

In [ ]:
comb_ann.DIAGNOSIS.fillna("NA").unique()

In [ ]:
comb_processes_scaled.T

In [ ]:
comb_ann["Platform_general"].unique()

In [ ]:
pca_plot(comb_processes_scaled.T, grouping=comb_ann["Major_group"], legend="out")

In [ ]:
umap_plot(comb_processes_scaled.T, grouping=comb_ann["Major_group"], legend="out")

### Compose a platform group column

In [ ]:
platform_mapping = pd.read_csv("platform_classification - sheet1.csv").set_index(
    "Platform ID"
)
platform_mapping_dict = platform_mapping["Broad Classification"].to_dict()
platform_mapping_dict.update({"RNASeq": "Illumina NGS"})
platform_mapping_dict.update({"Kassandra": "Illumina NGS"})

comb_ann["Platform_classified"] = comb_ann["Platform_general"].map(
    platform_mapping_dict
)
comb_ann["Platform_classified"].unique()

In [ ]:
pca_plot(
    comb_processes_scaled.T,
    grouping=comb_ann["Platform_classified"].fillna("na"),
    legend="out",
)

In [ ]:
umap_plot(
    comb_processes_scaled.T,
    grouping=comb_ann["Platform_classified"].fillna("na"),
    legend="out",
)

In [ ]:
pca_plot(
    comb_processes_scaled.T,
    grouping=comb_ann["Diagnosis_cell_type_general"],
    legend="out",
)
# ax.set_title('Expressions')

In [ ]:
pca_plot(
    comb_processes_scaled.T,
    grouping=comb_ann["Platform_general"].fillna("na"),
    legend="out",
)
# ax.set_title('Expressions')

### Try PCA/UMAP with expressions

In [ ]:
pca_plot(comb_exp.fillna(0), grouping=comb_ann["Major_group"], legend="out")

### Signatures analysis after median scaling withing each cohort

#### Scratch

In [ ]:
comb_ann["COHORT_LABEL"].fillna("na").value_counts()

In [ ]:
comb_ann.Cohort.fillna("na").value_counts()

In [ ]:
comb_ann["Cohort_general"] = comb_ann["COHORT_LABEL"]
comb_ann["Cohort_general"][comb_ann["COHORT_LABEL"].isna()] = "Kassandra"
comb_ann["Cohort_general"][~comb_ann["Cohort"].isna()] = comb_ann["Cohort"][
    ~comb_ann["Cohort"].isna()
]
cohorts_counts = comb_ann.Cohort_general.fillna("na").value_counts()
cohorts_counts

In [ ]:
signatures_local_scaled

In [ ]:
scaled_signatures_list = []
scaled_ann_list = []
for cohort in tqdm(cohorts_counts.index[cohorts_counts >= 100]):
    signatures_local = signatures[comb_ann.index[comb_ann.Cohort_general == cohort]]
    scaled_ann_list.append(comb_ann[comb_ann.Cohort_general == cohort])
    signatures_local_scaled = median_scale(
        data=signatures_local, axis=1
    )
    scaled_signatures_list.append(signatures_local_scaled)
signatures_scaled_by_cohort = pd.concat(scaled_signatures_list, axis=1)
ann_scaled_by_cohort = pd.concat(scaled_ann_list, axis=0)
signatures_scaled_by_cohort

In [ ]:
ann_scaled_by_cohort

In [ ]:
umap_plot(
    signatures_scaled_by_cohort.T.fillna(0).clip(10),
    grouping=ann_scaled_by_cohort["Major_group"],
    legend="out",
)

In [ ]:
dlbcl_gmt = read_gene_sets(os.path.join(FL_SHARED_ROOT, "MFP_REFERENCE", "dlbcl_signatures.gmt"))
signatures = ssgsea_formula(comb_exp.T, dlbcl_gmt)
progeny = run_progeny(comb_exp.T)

# comb_processes = pd.concat([signatures, progeny[progeny.index.isin(['NFkB', 'PI3K', 'p53'])]])

#:param axis: int, default=0, axis to be applied on: if 0, scale over columns, otherwise (if 1) scale over rows
comb_processes_scaled = median_scale(data=comb_processes, axis=1)

In [ ]:
comb_processes

### Clustermap of signatures

# internal B-cell typing analysis reproduced

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann

In [ ]:
comb_ann.Diagnosis_cell_type_general.fillna("na").value_counts()

Effector_cells, Follicular_dendritic_cells, Treg, Follicular_helper_T_cells, Lymphatic_endothelium, M2_signature, MHCII, Proliferation_rate



In [ ]:
# dlbcl_gmt = read_gene_sets(os.path.join(FL_SHARED_ROOT, "MFP_REFERENCE", "dlbcl_signatures.gmt"))
dlbcl_gmt = read_gene_sets("${FL_SHARED_ROOT}/projects/FL_META/FL_ME_genesets_upd_2024.gmt")
dlbcl_gmt

In [ ]:
dlbcl_gmt = read_gene_sets(os.path.join(FL_SHARED_ROOT, "MFP_REFERENCE", "dlbcl_signatures.gmt"))
signatures = ssgsea_formula(comb_exp.T, dlbcl_gmt)
progeny = run_progeny(comb_exp.T)

# SOM analysis of the cohort

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")

comb_ann = pd.read_csv(
    "comb_ann_unified.csv", sep=","
).set_index("Unnamed: 0")

In [ ]:
# Activate the pandas to R data frame conversion
pandas2ri.activate()


def batch_correct_for_som(data, annotation, batch_col="RNA_BATCH"):
    """
    Performs batch correction highly suitable for the SOM approach using limma::removeBatchEffect.
    SOM is sensitive to absolute expression shifts; limma effectively centers these
    shifts while maintaining biological structure.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations.
    batch_col : str
        Column containing the batch variables.

    Returns:
    --------
    pd.DataFrame
        Batch-corrected expression data.
    """
    print("Loading R package 'limma' for batch correction...")
    limma = importr("limma")
    base = importr("base")

    # Transpose data so Genes are rows and Samples are columns (standard R format)
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert to R Matrix
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    # Extract batches
    batches = annotation.loc[common_idx, batch_col].astype(str).values
    r_batches = ro.StrVector(batches)

    # Run limma::removeBatchEffect
    print(f"Applying batch correction for {batch_col}...")
    corrected_r = limma.removeBatchEffect(r_X_mat, batch=r_batches)

    # Convert back to pandas DataFrame and transpose back (Samples as rows)
    # rpy2 often auto-converts R matrices to numpy arrays when pandas2ri is active
    if isinstance(corrected_r, np.ndarray):
        corrected_matrix = corrected_r
    else:
        corrected_matrix = pandas2ri.rpy2py(corrected_r)

    corrected_df = pd.DataFrame(corrected_matrix, index=X.index, columns=X.columns).T

    return corrected_df


def run_opossom_analysis(
    data, annotation, group_col="Major_group", dataset_name="MyCohort"
):
    """
    Runs the full oposSOM pipeline natively in R via rpy2.
    It generates all SOM pictures, HTML reports, and tables in the current working directory.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns) - optimally batch-corrected.
    annotation : pd.DataFrame
        Sample annotations.
    group_col : str
        Biological grouping column for coloring and spot analysis.

    Returns:
    --------
    env : rpy2.robjects.environments.Environment
        The R environment containing the full oposSOM state.
    metagenes_df : pd.DataFrame
        Expression matrix of the SOM metagenes (Metagenes as rows, Samples as cols).
    bmu_df : pd.DataFrame
        Mapping of original Genes to their Best Matching Unit (Metagene).
    """
    print("Loading R package 'oposSOM'...")
    oposSOM = importr("oposSOM")
    base = importr("base")

    # Transpose to [Genes x Samples]
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert Data and Groups to R objects
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    groups = annotation.loc[common_idx, group_col].astype(str).values
    r_groups = ro.StrVector(groups)

    # Define oposSOM preferences
    # These trigger the full comprehensive analysis (portraits, spots, reports)
    pref = ro.ListVector(
        {
            "dataset.name": dataset_name,
            "dim.1stLvlSom": "automatic",
            "standard.spot.modules": "kmeans",
            "adjust.expression.values": False,  # Set to False if already batch-corrected
            "feature.centralization": True,
            "sample.quantile.normalization": True,
        }
    )

    print("Initializing oposSOM environment...")
    env = oposSOM.opossom_new(pref)

    # Inject data into the R environment
    env["indata"] = r_X_mat
    env["group.labels"] = r_groups

    # Execute the massive oposSOM run function
    print(
        "Running full oposSOM pipeline. This will generate PDFs and reports locally..."
    )
    oposSOM.opossom_run(env)

    # --- Extract Results Back to Python ---
    print("Extracting Metagene matrix and Gene mappings...")

    # 1. Metagene matrix
    metagenes_r = env["metadata"]
    metagenes_mat = (
        metagenes_r
        if isinstance(metagenes_r, np.ndarray)
        else pandas2ri.rpy2py(metagenes_r)
    )
    metagenes_df = pd.DataFrame(metagenes_mat, columns=X.columns)
    metagenes_df.index = [f"Metagene_{i+1}" for i in range(metagenes_df.shape[0])]

    # 2. Gene to Metagene mapping (Best Matching Unit)
    som_result = env["som.result"]
    bmu_r = som_result.rx2("feature.BMU")
    bmu_arr = bmu_r if isinstance(bmu_r, np.ndarray) else pandas2ri.rpy2py(bmu_r)
    bmu_df = pd.DataFrame({"Best_Matching_Unit": bmu_arr}, index=X.index)

    return env, metagenes_df, bmu_df


def cluster_metagenes(metagenes_df, n_clusters=15):
    """
    Performs unsupervised clustering of the extracted metagenes.

    Parameters:
    -----------
    metagenes_df : pd.DataFrame
        The metagene expression matrix (Metagenes as rows).
    n_clusters : int
        Number of super-clusters to form.

    Returns:
    --------
    pd.DataFrame
        The metagenes assigned to their unsupervised clusters.
    """
    print(f"Clustering {metagenes_df.shape[0]} metagenes into {n_clusters} clusters...")

    # Using Agglomerative (Hierarchical) Clustering which is standard for SOM nodes
    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters, metric="euclidean", linkage="ward"
    )
    labels = clusterer.fit_predict(metagenes_df.values)

    cluster_df = pd.DataFrame({"Metagene_Cluster": labels}, index=metagenes_df.index)

    return cluster_df


def retrieve_and_score_genesets(env, keywords=None):
    """
    Retrieves the full list of genesets used by oposSOM, allows filtering by keywords,
    and returns the sample-wise Gene Set Z-scores (GSZ) calculated during the run.

    Parameters:
    -----------
    env : rpy2.robjects.environments.Environment
        The oposSOM environment after running opossom_run.
    keywords : list of str, optional
        Only return genesets that contain these keywords (e.g., ['KEGG', 'B_CELL']).

    Returns:
    --------
    available_genesets : list
        List of geneset names matching the criteria.
    gsz_scores_df : pd.DataFrame
        The GSZ (Gene Set Z-score) matrix for the selected genesets across all samples.
    """
    print("Retrieving Gene Set Enrichment scores...")

    # Extract the full list of geneset names
    gs_list = env["gs.def.list"]
    all_genesets = list(gs_list.names)

    # Filter by keywords if provided
    if keywords:
        selected_genesets = [
            name
            for name in all_genesets
            if any(k.lower() in name.lower() for k in keywords)
        ]
    else:
        selected_genesets = all_genesets

    print(f"Found {len(selected_genesets)} genesets matching your criteria.")

    # Extract the GSZ scores matrix computed by oposSOM
    gsz_r = env["samples.GSZ.scores"]
    gsz_mat = gsz_r if isinstance(gsz_r, np.ndarray) else pandas2ri.rpy2py(gsz_r)
    gsz_df = pd.DataFrame(gsz_mat)

    # Assign row names (genesets) and column names (samples)
    gsz_df.index = list(gsz_r.rownames)
    gsz_df.columns = list(gsz_r.colnames)

    # Filter the dataframe to only include the selected genesets
    filtered_gsz_df = gsz_df.loc[selected_genesets]

    return selected_genesets, filtered_gsz_df


# ==========================================
# EXAMPLE USAGE PIPELINE
# ==========================================
# 1. Batch Correction
# corrected_exp = batch_correct_for_som(comp_exp, comb_ann, batch_col='RNA_BATCH')
#
# 2. Run oposSOM
# env, metagenes_matrix, gene_to_metagene_map = run_opossom_analysis(
#     corrected_exp, comb_ann, group_col='Diagnosis_cell_type_general'
# )
#
# 3. Unsupervised Clustering of Metagenes
# metagene_clusters = cluster_metagenes(metagenes_matrix, n_clusters=15)
#
# 4. Geneset Enrichment Scores Retrieval
# pathways, pathway_scores = retrieve_and_score_genesets(env, keywords=['HALLMARK', 'KEGG'])

In [ ]:
corrected_exp = batch_correct_for_som(
    filtered_exp, comb_ann_dedup, batch_col="RNA_BATCH"
)

In [ ]:
??run_opossom_analysis

In [ ]:
env, metagenes_matrix, gene_to_metagene_map = run_opossom_analysis(
    normalized_fsqn, comb_ann, group_col="Major_group"
)

# env.to_csv('env.csv')
metagenes_matrix.to_csv("metagenes_matrix.csv")
gene_to_metagene_map.to_csv("gene_to_metagene_map.csv")

In [ ]:
# 3. Unsupervised Clustering of Metagenes
metagene_clusters = cluster_metagenes(metagenes_matrix, n_clusters=15)
#
# 4. Geneset Enrichment Scores Retrieval
pathways, pathway_scores = retrieve_and_score_genesets(
    env, keywords=["HALLMARK", "KEGG"]
)

pathways.to_csv("pathways.csv")
pathway_scores.to_csv("pathway_scores.csv")

In [ ]:
corrected_exp

### Python script to run all the code for SOM with prior normalization

In [ ]:
import glob
import io
import itertools
import math
import os
import pickle
import subprocess
import warnings
from pathlib import Path
from statistics import median

import boto3
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import norm
from supervenn import supervenn
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

# Cohort assembly originally ran against an internal cohort database, configured
# here with an API key. The public pipeline reads prepared tables directly, so no
# client and no credential is needed.

import pyreadr
import rpy2.robjects as ro
from biomart import BiomartServer
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from scipy.interpolate import interp1d
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_ann.tsv"), sep="\t"
).set_index("Unnamed: 0")


bad_batches = ["GPL14951_FFPE_Unknown", "GPL8432_FFPE_Unknown", "GPL13938_FFPE_Unknown"]
bad_cohorts = ["SOM"]


# "GPL8432_FFPE_Unknown",
# "GPL6244_FFPE_Unknown",
# "GPL17586_FF_Unknown",
# "GPL20188_FF_Unknown",
# "GPL23541_FF_Unknown",
# "GPL887_FFPE_Unknown",
# "GPL26356_FF_Unknown",
# "GPL17047_FF_Unknown",
# "GPL6244_FF_Unknown",
# "GPL13938_FFPE_Unknown",
# ]
rare_groups = [
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
    "Other",
]


comb_ann = comb_ann[~comb_ann.RNA_BATCH.isin(bad_batches)]
comb_ann = comb_ann[~comb_ann.Major_group.isin(rare_groups)]
comb_ann = comb_ann[~comb_ann.COHORT_LABEL.isin(bad_cohorts)]

comb_ann.loc[
    comb_ann.Major_group.isin(["Normal_B_cells", "Kassandra"]),
    ["Major_group"],
] = "Normal_B_cells"


common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
filtered_exp = comb_exp.loc[common_samples].dropna(axis=1)
filtered_exp = filtered_exp.loc[~filtered_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
batch_groups = comb_ann.loc[common_samples, "RNA_BATCH"]
batch_groups = batch_groups.loc[~batch_groups.index.duplicated(keep="first"),]
comb_ann_dedup = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]


def fsqn_normalize(data, groups, target_group=None):
    """
    Implementation of Feature Specific Quantile Normalization (FSQN).
    FSQN transforms the distribution of each feature (gene) within each group
    to match a target distribution (either a specific group or the global mean).

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    groups : pd.Series or list
        The grouping variable (e.g., RNA_BATCH).
    target_group : str, optional
        The name of the group to use as the reference distribution.
        If None, the global average distribution across all samples is used.

    Returns:
    --------
    pd.DataFrame
        FSQN-normalized expression data.
    """
    # Align groups with data index
    if not isinstance(groups, pd.Series):
        groups = pd.Series(groups, index=data.index)
    groups = groups.loc[data.index]

    unique_groups = np.unique(groups)
    normalized_data = data.copy()

    # 1. Define the target distribution
    # FSQN usually targets the distribution of a 'reference' batch or the global mean
    if target_group and target_group in unique_groups:
        target_dist = np.sort(data.loc[groups == target_group].values, axis=1).mean(
            axis=0
        )
    else:
        # Use global mean distribution as target
        target_dist = np.sort(data.values, axis=1).mean(axis=0)

    n_genes = data.shape[1]
    # Create an interpolation line for the target distribution
    target_ranks = np.linspace(0, 1, n_genes)

    # 2. Normalize each group to the target
    for g in unique_groups:
        group_indices = data.index[groups == g]
        group_data = data.loc[group_indices].values

        for i in range(len(group_indices)):
            sample_values = group_data[i, :]
            # Get the rank order of values in the current sample
            order = np.argsort(sample_values)

            # Map the target distribution values onto the sample's ranks
            # This forces the sample to have the target distribution
            normalized_data.loc[group_indices[i]].iloc[order] = target_dist

    return normalized_data


normalized_fsqn = fsqn_normalize(filtered_exp, comb_ann_dedup["RNA_BATCH"])

# Activate the pandas to R data frame conversion
pandas2ri.activate()


def batch_correct_for_som(data, annotation, batch_col="RNA_BATCH"):
    """
    Performs batch correction highly suitable for the SOM approach using limma::removeBatchEffect.
    SOM is sensitive to absolute expression shifts; limma effectively centers these
    shifts while maintaining biological structure.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations.
    batch_col : str
        Column containing the batch variables.

    Returns:
    --------
    pd.DataFrame
        Batch-corrected expression data.
    """
    print("Loading R package 'limma' for batch correction...")
    limma = importr("limma")
    base = importr("base")

    # Transpose data so Genes are rows and Samples are columns (standard R format)
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert to R Matrix
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    # Extract batches
    batches = annotation.loc[common_idx, batch_col].astype(str).values
    r_batches = ro.StrVector(batches)

    # Run limma::removeBatchEffect
    print(f"Applying batch correction for {batch_col}...")
    corrected_r = limma.removeBatchEffect(r_X_mat, batch=r_batches)

    # Convert back to pandas DataFrame and transpose back (Samples as rows)
    # rpy2 often auto-converts R matrices to numpy arrays when pandas2ri is active
    if isinstance(corrected_r, np.ndarray):
        corrected_matrix = corrected_r
    else:
        corrected_matrix = pandas2ri.rpy2py(corrected_r)

    corrected_df = pd.DataFrame(corrected_matrix, index=X.index, columns=X.columns).T

    return corrected_df


def run_opossom_analysis(
    data, annotation, group_col="Major_group", dataset_name="MyCohort"
):
    """
    Runs the full oposSOM pipeline natively in R via rpy2.
    It generates all SOM pictures, HTML reports, and tables in the current working directory.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns) - optimally batch-corrected.
    annotation : pd.DataFrame
        Sample annotations.
    group_col : str
        Biological grouping column for coloring and spot analysis.

    Returns:
    --------
    env : rpy2.robjects.environments.Environment
        The R environment containing the full oposSOM state.
    metagenes_df : pd.DataFrame
        Expression matrix of the SOM metagenes (Metagenes as rows, Samples as cols).
    bmu_df : pd.DataFrame
        Mapping of original Genes to their Best Matching Unit (Metagene).
    """
    print("Loading R package 'oposSOM'...")
    oposSOM = importr("oposSOM")
    base = importr("base")

    # Transpose to [Genes x Samples]
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert Data and Groups to R objects
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    groups = annotation.loc[common_idx, group_col].astype(str).values
    r_groups = ro.StrVector(groups)

    # Define oposSOM preferences
    # These trigger the full comprehensive analysis (portraits, spots, reports)
    pref = ro.ListVector(
        {
            "dataset.name": dataset_name,
            "dim.1stLvlSom": "automatic",
            "standard.spot.modules": "kmeans",
            "adjust.expression.values": False,  # Set to False if already batch-corrected
            "feature.centralization": True,
            "sample.quantile.normalization": True,
        }
    )

    print("Initializing oposSOM environment...")
    env = oposSOM.opossom_new(pref)

    # Inject data into the R environment
    env["indata"] = r_X_mat
    env["group.labels"] = r_groups

    # Execute the massive oposSOM run function
    print(
        "Running full oposSOM pipeline. This will generate PDFs and reports locally..."
    )
    oposSOM.opossom_run(env)

    # --- Extract Results Back to Python ---
    print("Extracting Metagene matrix and Gene mappings...")

    # 1. Metagene matrix
    metagenes_r = env["metadata"]
    metagenes_mat = (
        metagenes_r
        if isinstance(metagenes_r, np.ndarray)
        else pandas2ri.rpy2py(metagenes_r)
    )
    metagenes_df = pd.DataFrame(metagenes_mat, columns=X.columns)
    metagenes_df.index = [f"Metagene_{i+1}" for i in range(metagenes_df.shape[0])]

    # 2. Gene to Metagene mapping (Best Matching Unit)
    som_result = env["som.result"]
    bmu_r = som_result.rx2("feature.BMU")
    bmu_arr = bmu_r if isinstance(bmu_r, np.ndarray) else pandas2ri.rpy2py(bmu_r)
    bmu_df = pd.DataFrame({"Best_Matching_Unit": bmu_arr}, index=X.index)

    return env, metagenes_df, bmu_df


def cluster_metagenes(metagenes_df, n_clusters=15):
    """
    Performs unsupervised clustering of the extracted metagenes.

    Parameters:
    -----------
    metagenes_df : pd.DataFrame
        The metagene expression matrix (Metagenes as rows).
    n_clusters : int
        Number of super-clusters to form.

    Returns:
    --------
    pd.DataFrame
        The metagenes assigned to their unsupervised clusters.
    """
    print(f"Clustering {metagenes_df.shape[0]} metagenes into {n_clusters} clusters...")

    # Using Agglomerative (Hierarchical) Clustering which is standard for SOM nodes
    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters, metric="euclidean", linkage="ward"
    )
    labels = clusterer.fit_predict(metagenes_df.values)

    cluster_df = pd.DataFrame({"Metagene_Cluster": labels}, index=metagenes_df.index)

    return cluster_df


def retrieve_and_score_genesets(env, keywords=None):
    """
    Retrieves the full list of genesets used by oposSOM, allows filtering by keywords,
    and returns the sample-wise Gene Set Z-scores (GSZ) calculated during the run.

    Parameters:
    -----------
    env : rpy2.robjects.environments.Environment
        The oposSOM environment after running opossom_run.
    keywords : list of str, optional
        Only return genesets that contain these keywords (e.g., ['KEGG', 'B_CELL']).

    Returns:
    --------
    available_genesets : list
        List of geneset names matching the criteria.
    gsz_scores_df : pd.DataFrame
        The GSZ (Gene Set Z-score) matrix for the selected genesets across all samples.
    """
    print("Retrieving Gene Set Enrichment scores...")

    # Extract the full list of geneset names
    gs_list = env["gs.def.list"]
    all_genesets = list(gs_list.names)

    # Filter by keywords if provided
    if keywords:
        selected_genesets = [
            name
            for name in all_genesets
            if any(k.lower() in name.lower() for k in keywords)
        ]
    else:
        selected_genesets = all_genesets

    print(f"Found {len(selected_genesets)} genesets matching your criteria.")

    # Extract the GSZ scores matrix computed by oposSOM
    gsz_r = env["samples.GSZ.scores"]
    gsz_mat = gsz_r if isinstance(gsz_r, np.ndarray) else pandas2ri.rpy2py(gsz_r)
    gsz_df = pd.DataFrame(gsz_mat)

    # Assign row names (genesets) and column names (samples)
    gsz_df.index = list(gsz_r.rownames)
    gsz_df.columns = list(gsz_r.colnames)

    # Filter the dataframe to only include the selected genesets
    filtered_gsz_df = gsz_df.loc[selected_genesets]

    return selected_genesets, filtered_gsz_df


env, metagenes_matrix, gene_to_metagene_map = run_opossom_analysis(
    normalized_fsqn, comb_ann, group_col="Major_group"
)

# env.to_csv('env.csv')
metagenes_matrix.to_csv("metagenes_matrix.csv")
gene_to_metagene_map.to_csv("gene_to_metagene_map.csv")

# 3. Unsupervised Clustering of Metagenes
metagene_clusters = cluster_metagenes(metagenes_matrix, n_clusters=15)
#
# 4. Geneset Enrichment Scores Retrieval
pathways, pathway_scores = retrieve_and_score_genesets(
    env, keywords=["HALLMARK", "KEGG"]
)

pathways.to_csv("pathways.csv")
pathway_scores.to_csv("pathway_scores.csv")

#### Python script for FSQN R to run SOM

In [ ]:
import glob
import io
import itertools
import math
import os
import pickle
import subprocess
import warnings
from pathlib import Path
from statistics import median

import boto3
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm
from supervenn import supervenn
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

import pyreadr
import rpy2.robjects as ro
from biomart import BiomartServer
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from scipy.interpolate import interp1d
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "fsqn_normalized_exp_R.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann_dedup = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "ann_normalized_fsqn_R.tsv"), sep="\t"
).set_index("Unnamed: 0")

# Activate the pandas to R data frame conversion
pandas2ri.activate()


def batch_correct_for_som(data, annotation, batch_col="RNA_BATCH"):
    """
    Performs batch correction highly suitable for the SOM approach using limma::removeBatchEffect.
    SOM is sensitive to absolute expression shifts; limma effectively centers these
    shifts while maintaining biological structure.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns).
    annotation : pd.DataFrame
        Sample annotations.
    batch_col : str
        Column containing the batch variables.

    Returns:
    --------
    pd.DataFrame
        Batch-corrected expression data.
    """
    print("Loading R package 'limma' for batch correction...")
    limma = importr("limma")
    base = importr("base")

    # Transpose data so Genes are rows and Samples are columns (standard R format)
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert to R Matrix
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    # Extract batches
    batches = annotation.loc[common_idx, batch_col].astype(str).values
    r_batches = ro.StrVector(batches)

    # Run limma::removeBatchEffect
    print(f"Applying batch correction for {batch_col}...")
    corrected_r = limma.removeBatchEffect(r_X_mat, batch=r_batches)

    # Convert back to pandas DataFrame and transpose back (Samples as rows)
    # rpy2 often auto-converts R matrices to numpy arrays when pandas2ri is active
    if isinstance(corrected_r, np.ndarray):
        corrected_matrix = corrected_r
    else:
        corrected_matrix = pandas2ri.rpy2py(corrected_r)

    corrected_df = pd.DataFrame(corrected_matrix, index=X.index, columns=X.columns).T

    return corrected_df


def run_opossom_analysis(
    data, annotation, group_col="Major_group", dataset_name="MyCohort"
):
    """
    Runs the full oposSOM pipeline natively in R via rpy2.
    It generates all SOM pictures, HTML reports, and tables in the current working directory.

    Parameters:
    -----------
    data : pd.DataFrame
        Gene expression data (Samples as rows, Genes as columns) - optimally batch-corrected.
    annotation : pd.DataFrame
        Sample annotations.
    group_col : str
        Biological grouping column for coloring and spot analysis.

    Returns:
    --------
    env : rpy2.robjects.environments.Environment
        The R environment containing the full oposSOM state.
    metagenes_df : pd.DataFrame
        Expression matrix of the SOM metagenes (Metagenes as rows, Samples as cols).
    bmu_df : pd.DataFrame
        Mapping of original Genes to their Best Matching Unit (Metagene).
    """
    print("Loading R package 'oposSOM'...")
    oposSOM = importr("oposSOM")
    base = importr("base")

    # Transpose to [Genes x Samples]
    X = data.T
    common_idx = X.columns.intersection(annotation.index)
    X = X[common_idx]

    # Convert Data and Groups to R objects
    r_X = pandas2ri.py2rpy(X)
    r_X_mat = base.as_matrix(r_X)

    groups = annotation.loc[common_idx, group_col].astype(str).values
    r_groups = ro.StrVector(groups)

    # Define oposSOM preferences
    # These trigger the full comprehensive analysis (portraits, spots, reports)
    pref = ro.ListVector(
        {
            "dataset.name": dataset_name,
            "dim.1stLvlSom": "automatic",
            "standard.spot.modules": "kmeans",
            "adjust.expression.values": False,  # Set to False if already batch-corrected
            "feature.centralization": True,
            "sample.quantile.normalization": True,
            # 'geneset.analysis': True,
            # 'psf.analysis': True,
            #
        }
    )

    print("Initializing oposSOM environment...")
    env = oposSOM.opossom_new(pref)

    # Inject data into the R environment
    env["indata"] = r_X_mat
    env["group.labels"] = r_groups

    # Execute the massive oposSOM run function
    print(
        "Running full oposSOM pipeline. This will generate PDFs and reports locally..."
    )
    oposSOM.opossom_run(env)

    # --- Extract Results Back to Python ---
    print("Extracting Metagene matrix and Gene mappings...")

    # 1. Metagene matrix
    metagenes_r = env["metadata"]
    metagenes_mat = (
        metagenes_r
        if isinstance(metagenes_r, np.ndarray)
        else pandas2ri.rpy2py(metagenes_r)
    )
    metagenes_df = pd.DataFrame(metagenes_mat, columns=X.columns)
    metagenes_df.index = [f"Metagene_{i+1}" for i in range(metagenes_df.shape[0])]

    # 2. Gene to Metagene mapping (Best Matching Unit)
    som_result = env["som.result"]
    bmu_r = som_result.rx2("feature.BMU")
    bmu_arr = bmu_r if isinstance(bmu_r, np.ndarray) else pandas2ri.rpy2py(bmu_r)
    bmu_df = pd.DataFrame({"Best_Matching_Unit": bmu_arr}, index=X.index)

    return env, metagenes_df, bmu_df


def cluster_metagenes(metagenes_df, n_clusters=15):
    """
    Performs unsupervised clustering of the extracted metagenes.

    Parameters:
    -----------
    metagenes_df : pd.DataFrame
        The metagene expression matrix (Metagenes as rows).
    n_clusters : int
        Number of super-clusters to form.

    Returns:
    --------
    pd.DataFrame
        The metagenes assigned to their unsupervised clusters.
    """
    print(f"Clustering {metagenes_df.shape[0]} metagenes into {n_clusters} clusters...")

    # Using Agglomerative (Hierarchical) Clustering which is standard for SOM nodes
    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters, metric="euclidean", linkage="ward"
    )
    labels = clusterer.fit_predict(metagenes_df.values)

    cluster_df = pd.DataFrame({"Metagene_Cluster": labels}, index=metagenes_df.index)

    return cluster_df


def retrieve_and_score_genesets(env, keywords=None):
    """
    Retrieves the full list of genesets used by oposSOM, allows filtering by keywords,
    and returns the sample-wise Gene Set Z-scores (GSZ) calculated during the run.

    Parameters:
    -----------
    env : rpy2.robjects.environments.Environment
        The oposSOM environment after running opossom_run.
    keywords : list of str, optional
        Only return genesets that contain these keywords (e.g., ['KEGG', 'B_CELL']).

    Returns:
    --------
    available_genesets : list
        List of geneset names matching the criteria.
    gsz_scores_df : pd.DataFrame
        The GSZ (Gene Set Z-score) matrix for the selected genesets across all samples.
    """
    print("Retrieving Gene Set Enrichment scores...")

    # Extract the full list of geneset names
    gs_list = env["gs.def.list"]
    all_genesets = list(gs_list.names)

    # Filter by keywords if provided
    if keywords:
        selected_genesets = [
            name
            for name in all_genesets
            if any(k.lower() in name.lower() for k in keywords)
        ]
    else:
        selected_genesets = all_genesets

    print(f"Found {len(selected_genesets)} genesets matching your criteria.")

    # Extract the GSZ scores matrix computed by oposSOM
    gsz_r = env["samples.GSZ.scores"]
    gsz_mat = gsz_r if isinstance(gsz_r, np.ndarray) else pandas2ri.rpy2py(gsz_r)
    gsz_df = pd.DataFrame(gsz_mat)

    # Assign row names (genesets) and column names (samples)
    gsz_df.index = list(gsz_r.rownames)
    gsz_df.columns = list(gsz_r.colnames)

    # Filter the dataframe to only include the selected genesets
    filtered_gsz_df = gsz_df.loc[selected_genesets]

    return selected_genesets, filtered_gsz_df


env, metagenes_matrix, gene_to_metagene_map = run_opossom_analysis(
    comb_exp, comb_ann_dedup, group_col="Diagnosis_cell_type_unified"
)

# env.to_csv('env.csv')
metagenes_matrix.to_csv("metagenes_matrix.csv")
gene_to_metagene_map.to_csv("gene_to_metagene_map.csv")

# 3. Unsupervised Clustering of Metagenes
metagene_clusters = cluster_metagenes(metagenes_matrix, n_clusters=15)
#
# 4. Geneset Enrichment Scores Retrieval
pathways, pathway_scores = retrieve_and_score_genesets(
    env, keywords=["HALLMARK", "KEGG"]
)

pathways.to_csv("pathways.csv")
pathway_scores.to_csv("pathway_scores.csv")

In [ ]:
comb_ann_dedup.Diagnosis_cell_type_unified

## Test batch effect of limma normalized dataset

In [ ]:
column = "RNA_BATCH"
palette = rna_batch_palette  # cohort_palette #rna_batch_palette # # #platform_palette #
# filtering = comb_ann_dedup.RNA_BATCH.isin(['GPL570_Unknown_Unknown', 'RNASeq_FFPE_Exome_capture'])
filtering = ~comb_ann_dedup.RNA_BATCH.isna()

figure, axs = plt.subplots(
    nrows=1, ncols=2, sharex=False, sharey=False, squeeze=True, figsize=(10, 5)
)

pca_plot(
    filtered_exp[filtering],
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="Non-normalized",
    palette=palette,
)

pca_plot(
    corrected_exp[
        filtering
    ],  # harmony_normalized_exp# normalized_exp, #normalized_fsqn #
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="Normalized",
    palette=palette,
)

In [ ]:
column = "PLATFORM_RNA"
palette = platform_palette  # rna_batch_palette

figure, axs = plt.subplots(
    nrows=1,
    ncols=2,
    sharex=False,
    sharey=False,
    squeeze=True,
    figsize=(10, 5),
)

umap_plot(
    filtered_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="Non-normalized",
    palette=palette,
)

umap_plot(
    np.log2(normalized_fsqn + 1),  # normalized_exp,
    grouping=comb_ann_dedup[column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="Normalized",
    palette=palette,
)

## SOM RData analysis

### Extract metagenes df

In [ ]:
# Activate pandas conversion
pandas2ri.activate()

# Path to your massive RData file
rdata_path = "./MyCohort+ - Results FSQN R/MyCohort+.RData"

print(f"Loading {rdata_path} natively via R...")
# This replicates the R command: load("./MyCohort+ - Results/MyCohort+.RData")
ro.r['load'](rdata_path)

# When you use load() in R, it places the saved variables into the global environment.
# Let's check what variables were loaded (oposSOM usually saves it as 'env')
loaded_vars = list(ro.globalenv.keys())
print("Variables loaded from RData:", loaded_vars)

# Assuming the saved object is named 'env' (which is the default in oposSOM)
if 'env' in ro.globalenv:
    loaded_env = ro.globalenv['env']
    
    # Now you can extract data from it exactly as you did in the pipeline function!
    print("Extracting metadata (metagenes) from the loaded environment...")
    
    metagenes_r = loaded_env['metadata']
    
    # Handle the rpy2 auto-conversion to numpy array gracefully
    metagenes_mat = metagenes_r if isinstance(metagenes_r, np.ndarray) else pandas2ri.rpy2py(metagenes_r)
    
    # Convert back to pandas DataFrame
    metagenes_df = pd.DataFrame(metagenes_mat)
    
    # If the R matrix had row/col names, you can extract them like this:
    if hasattr(metagenes_r, 'rownames') and metagenes_r.rownames is not ro.NULL:
        metagenes_df.index = list(metagenes_r.rownames)
    if hasattr(metagenes_r, 'colnames') and metagenes_r.colnames is not ro.NULL:
        metagenes_df.columns = list(metagenes_r.colnames)
        
    print(metagenes_df.head())
else:
    print("Could not find 'env' in the loaded RData file.")

metagenes_df.columns = comb_ann_dedup.index

### Extract all data

In [ ]:
def extract_r_object(r_obj):
    """
    Recursively converts R objects (Lists, Matrices, Vectors, DataFrames) 
    into native Python/Pandas structures.
    """
    # 1. Handle NULLs
    # FIX: Using 'is' instead of '==' prevents NumPy element-wise comparison errors
    if r_obj is ro.NULL or type(r_obj).__name__ == 'NULLType':
        return None

    # 2. Skip R Functions/Environments to avoid parsing errors
    if isinstance(r_obj, (ro.functions.Function, ro.environments.Environment)):
        return "<R_Function_or_Environment>"

    # 3. Handle R ListVectors (recursive dictionary conversion)
    # This handles nested structures like 'som.result' and 'spot.list.kmeans'
    if isinstance(r_obj, ListVector):
        res_dict = {}
        # R lists can be named or unnamed. If unnamed, use indices.
        names = r_obj.names if r_obj.names is not ro.NULL else [f"idx_{i}" for i in range(len(r_obj))]
        for k, v in zip(names, r_obj):
            res_dict[k] = extract_r_object(v)
        return res_dict

    # 4. Handle explicit R Matrices
    if isinstance(r_obj, Matrix):
        mat = np.array(r_obj)
        df = pd.DataFrame(mat)
        if hasattr(r_obj, 'rownames') and r_obj.rownames is not ro.NULL:
            df.index = list(r_obj.rownames)
        if hasattr(r_obj, 'colnames') and r_obj.colnames is not ro.NULL:
            df.columns = list(r_obj.colnames)
        return df

    # 5. Handle pre-converted numpy arrays (pandas2ri often converts matrices automatically)
    if isinstance(r_obj, np.ndarray):
        df = pd.DataFrame(r_obj)
        if hasattr(r_obj, 'rownames') and r_obj.rownames is not ro.NULL:
            df.index = list(r_obj.rownames)
        if hasattr(r_obj, 'colnames') and r_obj.colnames is not ro.NULL:
            df.columns = list(r_obj.colnames)
        return df

    # 6. Handle R DataFrames
    if isinstance(r_obj, DataFrame):
        return pandas2ri.rpy2py(r_obj)

    # 7. Handle R Atomic Vectors (Numeric, String, Bool)
    if isinstance(r_obj, (FloatVector, IntVector, StrVector, BoolVector)):
        val_list = list(r_obj)
        # If it's a named vector, make it a pandas Series (e.g., indata.sample.mean)
        if hasattr(r_obj, 'names') and r_obj.names is not ro.NULL:
            return pd.Series(val_list, index=list(r_obj.names))
        else:
            # If length 1, return the scalar (e.g., boolean flags)
            if len(val_list) == 1:
                return val_list[0]
            # Otherwise return a standard python list
            return val_list

    # 8. Fallback for unhandled types
    try:
        converted = pandas2ri.rpy2py(r_obj)
        if isinstance(converted, np.ndarray):
            return pd.DataFrame(converted)
        return converted
    except Exception:
        return f"<Unparsed R Object: {type(r_obj)}>"


# ==========================================
# Run the Extraction
# ==========================================
# Path to your massive RData file
#rdata_path = "./MyCohort+ - Results FSQN R/MyCohort+.RData"
rdata_path = "./MyCohort+ - Results Quantile norm/MyCohort+.RData"

print(f"Loading {rdata_path} natively via R...")
ro.r['load'](rdata_path)

if 'env' not in ro.globalenv:
    raise RuntimeError("Could not find 'env' in the loaded RData file.")

loaded_env = ro.globalenv['env']
loaded_vars = list(loaded_env.keys())

opossom_data = {}
print(f"Extracting {len(loaded_vars)} variables. This may take a moment for large lists...")

for key in tqdm(loaded_vars):
    # Skip functions to save time
    if key in ["csv.function"]:
        continue
    
    try:
        opossom_data[key] = extract_r_object(loaded_env[key])
    except Exception as e:
        print(f"Warning: Failed to extract '{key}' due to {str(e)}")
        opossom_data[key] = None

indata_df = pd.DataFrame(opossom_data['indata'])
indata_df.columns = comb_ann_dedup.index
indata_df.index = filtered_exp.columns

metagenes_df = pd.DataFrame(opossom_data['metadata'])
metagenes_df.columns = comb_ann_dedup.index

# visualize these metagenes for all spots on the individual profiles heatmaps
gene_to_metagene = opossom_data['som.result']['feature.BMU']
spot_signatures = opossom_data['spot.list.kmeans']['spots']
spot_signatures_up = opossom_data['spot.list.overexpression']['spots']
spot_signatures_down = opossom_data['spot.list.underexpression']['spots']
spot_signatures_FL_DLBCL = opossom_data['spot.list.group.overexpression']['spots']
spots_correlation_metagenes = opossom_data['spot.list.correlation']['spots']
spots_boundaries_metagenes = opossom_data['spot.list.dmap']['spots']

sample_labels = opossom_data['group.labels']

print("Extraction complete!")

In [ ]:
opossom_data.keys()

In [ ]:
opossom_data['files.name']

In [ ]:
opossom_data['spot.list.kmeans']['spots'].keys()

#### Documentation


In [ ]:
opossom_data[]

In [ ]:
list(loaded_env.keys())

In [ ]:
loaded_env['spot.list.dmap']

In [ ]:
metagenes_df

In [ ]:
comb_ann_dedup

In [ ]:
metagenes_df.columns = comb_ann_dedup.index
metagenes_df

### PCA/UMAP/tSNE

In [ ]:
Platform_general                                                                            6577
Diagnosis_cell_type                                                                         6593
Cohort_general                                                                              6604
TUMOR_NORMAL                                                                                6604
SAMPLE_NAME                                                                                 6604
RNA_BATCH                                                                                   6604
COHORT_LABEL                                                                                6604
Diagnosis_cell_type_general                                                                 6604
PLATFORM_RNA                                                                                6604
RNASEQ_SOURCE                                                                               6604
Major_group                                                                                 6604

In [ ]:
comb_ann_normal = comb_ann_dedup[comb_ann_dedup.Major_group.isin(['Kassandra', 'Normal_B_cells'])]
pd.crosstab(comb_ann_normal['RNA_BATCH'], comb_ann_normal['Diagnosis_unified_with_coo']).T

In [ ]:
print(pd.DataFrame(comb_ann_normal[['RNA_BATCH', 'Diagnosis_unified_with_coo']].value_counts()).to_string())

In [ ]:
comb_ann_dedup.Major_group.unique()

In [ ]:
filtered_exp

In [ ]:
column = "RNA_BATCH"#Diagnosis_unified_with_coo"
palette = rna_batch_palette  #lymphoma_ontogeny_palette # cohort_palette #rna_batch_palette # # #platform_palette #
filtering = ~comb_ann_dedup.RNA_BATCH.isna()
# filtering = comb_ann_dedup.RNA_BATCH.isin(['GPL570_Unknown_Unknown', 'RNASeq_FFPE_Exome_capture'])
#filtering = comb_ann_dedup.Major_group.isin(['Kassandra', 'Normal_B_cells'])

figure, axs = plt.subplots(
    nrows=1, ncols=2, sharex=False, sharey=False, squeeze=True, figsize=(10, 5)
)

pca_plot(
    np.log2(normalized_quantile[filtering]+1),
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="FSQN expressions",
    palette=palette,
)

pca_plot(
    metagenes_df.T[filtering],  # normalized_exp, #normalized_fsqn
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="SOM metagenes",
    palette=palette,
)

In [ ]:
column = "RNA_BATCH"#Diagnosis_unified_with_coo"
palette = rna_batch_palette  #lymphoma_ontogeny_palette # cohort_palette #rna_batch_palette # # #platform_palette #
filtering = ~comb_ann_dedup.RNA_BATCH.isna()
# filtering = comb_ann_dedup.RNA_BATCH.isin(['GPL570_Unknown_Unknown', 'RNASeq_FFPE_Exome_capture'])
#filtering = comb_ann_dedup.Major_group.isin(['Kassandra', 'Normal_B_cells'])


figure, axs = plt.subplots(
    nrows=1, ncols=2, sharex=False, sharey=False, squeeze=True, figsize=(10, 5)
)

umap_plot(
    np.log2(normalized_quantile[filtering]+1),
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="FSQN expressions",
    palette=palette,
)

umap_plot(
    metagenes_df.T[filtering],  # normalized_exp, #normalized_fsqn
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="SOM metagenes",
    palette=palette,
)

In [ ]:
column = "RNA_BATCH"#Diagnosis_unified_with_coo"
palette = rna_batch_palette  #lymphoma_ontogeny_palette # cohort_palette #rna_batch_palette # # #platform_palette #
filtering = ~comb_ann_dedup.RNA_BATCH.isna()
# filtering = comb_ann_dedup.RNA_BATCH.isin(['GPL570_Unknown_Unknown', 'RNASeq_FFPE_Exome_capture'])
#filtering = comb_ann_dedup.Major_group.isin(['Kassandra', 'Normal_B_cells'])


figure, axs = plt.subplots(
    nrows=1, ncols=2, sharex=False, sharey=False, squeeze=True, figsize=(10, 5)
)

tsne_plot(
    np.log2(normalized_quantile[filtering]+1),
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend=None,
    ax=axs.flatten()[0],
    title="FSQN expressions",
    palette=palette,
)

tsne_plot(
    metagenes_df.T[filtering],  # normalized_exp, #normalized_fsqn
    grouping=comb_ann_dedup[filtering][column].fillna("na"),
    legend="out",
    ax=axs.flatten()[1],
    title="SOM metagenes",
    palette=palette,
)

In [ ]:
list(loaded_env.keys())

### SOM visualization via heatmaps

the original pictures, both for individual samples and for entire large groups - including B cell types, diagnosis types etc.

In [ ]:
def create_som_colormap():
    """
    Creates the classic blue-to-green-to-yellow-to-red rainbow colormap 
    frequently used in oposSOM and traditional SOM visualizations.
    """
    colors = ['darkblue', 'blue', 'cyan', 'green', 'yellow', 'red', 'darkred']
    return LinearSegmentedColormap.from_list("som_rainbow", colors)

def plot_som_portrait(
    metagenes_df,
    samples,
    spots=None,
    grid_shape=None,
    cmap=None,
    figsize=(8, 8),
    title=None,
    vmin=None,
    vmax=None,
    spot_edge_color='black',
    spot_line_width=1.5,
    font_size=12,
    order='F',
    origin='lower'
):
    """
    Draws a classical SOM heatmap/portrait. Can visualize a single sample 
    or the average expression of a group of samples. Allows highlighting of 
    specific spots (modules) with borders and labels.

    Parameters:
    -----------
    metagenes_df : pd.DataFrame
        The metagene expression matrix (Metagenes as rows, Samples as columns).
    samples : str or list of str
        A single sample name, or a list of sample names. If a list is provided, 
        the average expression across these samples will be visualized.
    spots : dict, optional
        A dictionary mapping spot names to lists of metagenes. 
        Example: {'Spot A': ['Metagene_1', 'Metagene_10'], ...}
    grid_shape : tuple, optional
        The (rows, columns) of the SOM grid. If None, it assumes a perfect square.
    cmap : matplotlib Colormap, optional
        The colormap to use. Defaults to the classic SOM rainbow.
    figsize : tuple, optional
        Size of the matplotlib figure.
    title : str, optional
        Title of the plot.
    vmin, vmax : float, optional
        Minimum and maximum values for the colormap scaling. 
    spot_edge_color : str, optional
        Color of the borders drawn around highlighted spot cells.
    spot_line_width : float, optional
        Thickness of the spot borders.
    font_size : int, optional
        Font size for the spot labels and titles.
    order : str, optional
        Reshaping order. 'F' (Fortran/column-major) is the R standard used by oposSOM.
    origin : str, optional
        'lower' places Metagene_1 at the bottom-left. 'upper' places it at top-left.
        
    Returns:
    --------
    fig, ax : matplotlib Figure and Axes objects
    """
    
    # 1. Calculate 1D data (Single Sample vs Group Average)
    if isinstance(samples, list):
        data_1d = metagenes_df[samples].mean(axis=1).values
        default_title = f"Group Average SOM Portrait (n={len(samples)})"
    else:
        data_1d = metagenes_df[samples].values
        default_title = f"SOM Portrait: {samples}"
        
    if title is None:
        title = default_title

    # 2. Determine Grid Shape
    n_metagenes = len(data_1d)
    if grid_shape is None:
        side = int(np.sqrt(n_metagenes))
        if side * side != n_metagenes:
            raise ValueError(f"Number of metagenes ({n_metagenes}) is not a perfect square. Please provide grid_shape.")
        grid_shape = (side, side)

    # 3. Reshape to 2D
    # R normally stores data column-wise, so we use order='F' to reconstruct the map
    data_2d = data_1d.reshape(grid_shape, order=order)

    # 4. Set up Plotting
    if cmap is None:
        cmap = create_som_colormap()

    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot the heatmap
    im = ax.imshow(
        data_2d, 
        cmap=cmap, 
        origin=origin, 
        vmin=vmin, 
        vmax=vmax,
        interpolation='nearest' # 'nearest' keeps the discrete pixel look
    )
    
    # Optional: Add colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel('Expression', rotation=270, labelpad=15, fontsize=font_size)

    # 5. Highlight Spots
    if spots:
        # Pre-map all metagene names to their 1D integer index for fast lookup
        metagene_indices = {name: idx for idx, name in enumerate(metagenes_df.index)}
        
        for spot_name, metagene_list in spots.items():
            coords = []
            for mg in metagene_list:
                if mg not in metagene_indices:
                    continue
                idx = metagene_indices[mg]
                
                # Convert 1D index to 2D coordinates based on the reshaping order
                if order == 'F':
                    row = idx % grid_shape[0]
                    col = idx // grid_shape[0]
                else:
                    row = idx // grid_shape[1]
                    col = idx % grid_shape[1]
                
                coords.append((col, row))
                
                # Draw the cell boundary
                # In imshow, the pixel center is at integer coordinates. 
                # The bottom-left corner of the pixel is at (col - 0.5, row - 0.5)
                rect = patches.Rectangle(
                    (col - 0.5, row - 0.5), 1, 1,
                    linewidth=spot_line_width,
                    edgecolor=spot_edge_color,
                    facecolor='none',
                    zorder=10
                )
                ax.add_patch(rect)
            
            # Draw Spot Label at the centroid of the spot
            if coords:
                cols, rows = zip(*coords)
                centroid_col = sum(cols) / len(cols)
                centroid_row = sum(rows) / len(rows)
                
                # Add text with a small white stroke/bbox to ensure visibility over any color
                ax.text(
                    centroid_col, centroid_row, 
                    spot_name, 
                    color='black',
                    fontsize=font_size,
                    fontweight='bold',
                    ha='center', va='center',
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'),
                    zorder=20
                )

    # 6. Formatting
    ax.set_title(title, fontsize=font_size + 2, fontweight='bold', pad=15)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False) # Remove outer axes box for a cleaner look

    plt.tight_layout()
    return fig, ax

# ==========================================
# EXAMPLE USAGE
# ==========================================
# Assuming `opossom_data` is your extracted dictionary and `metagenes_df` is your matrix.
#
# 1. Visualize a single sample
# fig, ax = plot_som_portrait(metagenes_df, samples='GSM4644254-PUB_DLBCL_CARTFARAMAND')
# plt.show()
#
# 2. Visualize a group average (e.g., all DLBCL samples)
# dlbcl_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_cell_type_unified'] == 'Diffuse_Large_B_Cell_Lymphoma'].index.tolist()
#
# 3. Extract spot metagenes from opossom_data to highlight
# # The oposSOM 'spot.list.kmeans' structure typically holds metagene names under 'genes' or 'metagenes'
# my_spots = {}
# kmeans_data = opossom_data.get('spot.list.kmeans', {})
# for spot_id, spot_info in kmeans_data.items():
#     # In R, vectors of names are often returned as simple lists or arrays
#     if 'metagenes' in spot_info:
#         my_spots[spot_id] = list(spot_info['metagenes'])
#
# fig, ax = plot_som_portrait(
#     metagenes_df=metagenes_df,
#     samples=dlbcl_samples,
#     spots=my_spots,
#     title="DLBCL Average Portrait with K-Means Modules",
#     figsize=(10, 10),
#     spot_edge_color='black',
#     spot_line_width=2.0
# )
# plt.show()

In [ ]:
metagenes_df

In [ ]:
comb_ann_dedup['Major_group'].value_counts()

In [ ]:
comb_ann_dedup['Diagnosis_cell_type_unified'].value_counts()

In [ ]:
comb_ann_dedup['Diagnosis_unified_with_coo'].value_counts()

In [ ]:
comb_ann_normal = comb_ann_dedup[comb_ann_dedup.Major_group.isin(['Normal_B_cells', 'Kassandra'])]

print(comb_ann_normal[['Diagnosis_unified_with_coo', 'RNA_BATCH']].value_counts().to_string())

#### Major lymphoma subtypes

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_cell_type_unified'] == 'Diffuse_Large_B_Cell_Lymphoma'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples)
plt.show()

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_cell_type_unified'] == 'Follicular_Lymphoma'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples)
plt.show()

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_unified_with_coo'] == 'Diffuse_Large_B_Cell_Lymphoma_GCB'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples,
                            vmin = -15, vmax = 10
                           )
plt.show()

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_unified_with_coo'] == 'Diffuse_Large_B_Cell_Lymphoma_ABC'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples,
                           vmin = -15, vmax = 10
                           )
plt.show()

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Diagnosis_unified_with_coo'] == 'Diffuse_Large_B_Cell_Lymphoma_'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples,
                           vmin = -15, vmax = 10
                           )
plt.show()

#### Normal B cells by datasets

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Major_group'] == 'Normal_B_cells'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples,
                           vmin = -10, vmax = 10,
                           figsize=(5, 5)
                           )
plt.show()

In [ ]:
selected_samples = comb_ann_dedup[comb_ann_dedup['Major_group'] == 'Kassandra'].index.tolist()

fig, ax = plot_som_portrait(metagenes_df, samples=selected_samples,
                            #vmin = -100, vmax = 100,
                            figsize=(5, 5)
                           )
plt.show()

show the same cell types across different datasets and platforms; build a plt.subplots grid and box the profiles that group together by biology or by batch



#### Normal B cells by RNA batches and cell types

In [ ]:
def plot_som_grid(
    metagenes_df, 
    annotation_df, 
    cell_type_col='Diagnosis_unified_with_coo', 
    batch_col='RNA_BATCH',
    lymphoma_ontogeny_palette=None,
    rna_batch_palette=None,
    figsize_per_cell=(2, 2),
    vmin=None,
    vmax=None
):
    """
    Plots a large grid of SOM portraits organized by cell type (rows) and RNA batch (columns).
    Limits for colormaps are calculated per cell type and centered at zero.
    Includes color annotations for rows and columns.
    """
    # 1. Define logical sorting for B-cell ontogeny
    # Adjust this list as needed to perfectly match your desired biological progression
    logical_order = [
        'Bone_marrow_CD19+_', 
        'Immature_', 
        'Naive_', 
        'GC_', 
        'Centroblast_', 
        'Centrocyte_', 
        'Follicular_Lymphoma_', # Placed near GC origin
        'MZ_', 
        'Memory_', 
        'Plasmablast_', 
        'Plasma_', 
        'B_cells_' # Catch-all
    ]
    
    # Extract unique valid cell types and sort them
    available_cell_types = annotation_df[cell_type_col].unique()
    sorted_cell_types = [ct for ct in logical_order if ct in available_cell_types]
    # Append any unexpected cell types at the end
    sorted_cell_types += [ct for ct in available_cell_types if ct not in sorted_cell_types]
    
    # Extract unique valid batches and sort them by descending number of unique cell types
    valid_ann = annotation_df.dropna(subset=[batch_col])
    batch_ct_counts = valid_ann.groupby(batch_col)[cell_type_col].nunique()
    sorted_batches = batch_ct_counts.sort_values(ascending=False).index.tolist()
    
    nrows = len(sorted_cell_types)
    ncols = len(sorted_batches)
    
    # 2. Determine SOM Grid Shape
    n_metagenes = metagenes_df.shape[0]
    side = int(np.sqrt(n_metagenes))
    if side * side != n_metagenes:
        raise ValueError(f"Number of metagenes ({n_metagenes}) is not a perfect square.")
    som_shape = (side, side)
    
    # 3. Pre-calculate row-wise vmin/vmax centered at zero
    row_limits = {}
    grid_data = {} # Cache averages to avoid recomputing
    
    for ct in sorted_cell_types:
        ct_max_abs = 0
        grid_data[ct] = {}
        for b in sorted_batches:
            # Find samples that match both cell type and batch
            mask = (annotation_df[cell_type_col] == ct) & (annotation_df[batch_col] == b)
            samples = annotation_df.index[mask].intersection(metagenes_df.columns)
            
            if len(samples) > 0:
                # Calculate average
                avg_expr = metagenes_df[samples].mean(axis=1).values
                grid_data[ct][b] = avg_expr
                
                # Update max absolute value for this row
                max_val = np.max(np.abs(avg_expr))
                if max_val > ct_max_abs:
                    ct_max_abs = max_val
            else:
                grid_data[ct][b] = None
                
        row_limits[ct] = ct_max_abs

    # 4. Set up Plot and GridSpec
    # Add +1 row and column for the color annotations
    fig_width = ncols * figsize_per_cell[0] + 1
    fig_height = nrows * figsize_per_cell[1] + 1
    fig = plt.figure(figsize=(fig_width, fig_height))
    
    gs = gridspec.GridSpec(
        nrows + 1, ncols + 1, 
        width_ratios=[0.2] + [1]*ncols, 
        height_ratios=[0.2] + [1]*nrows,
        wspace=0.05, hspace=0.05
    )
    
    cmap = create_som_colormap()
    
    # 5. Draw Top Annotation (RNA Batches)
    for j, b in enumerate(sorted_batches):
        ax_top = fig.add_subplot(gs[0, j+1])
        color = rna_batch_palette.get(b, 'lightgray') if rna_batch_palette else 'gray'
        ax_top.set_facecolor(color)
        ax_top.set_xticks([])
        ax_top.set_yticks([])
        # Add angled text label
        ax_top.text(0.5, 0.5, b, rotation=45, ha='left', va='bottom', fontsize=10, transform=ax_top.transAxes)

    # 6. Draw Left Annotation (Cell Types)
    for i, ct in enumerate(sorted_cell_types):
        ax_left = fig.add_subplot(gs[i+1, 0])
        color = lymphoma_ontogeny_palette.get(ct, 'lightgray') if lymphoma_ontogeny_palette else 'gray'
        ax_left.set_facecolor(color)
        ax_left.set_xticks([])
        ax_left.set_yticks([])
        # Add cell type label
        ax_left.text(-0.2, 0.5, ct, ha='right', va='center', fontsize=12, transform=ax_left.transAxes)

    # 7. Draw SOM Grids
    for i, ct in enumerate(sorted_cell_types):
        if not vmax:
            vmax = row_limits[ct]
        if not vmin:
            vmin = -vmax
        
        for j, b in enumerate(sorted_batches):
            ax = fig.add_subplot(gs[i+1, j+1])
            data_1d = grid_data[ct][b]
            
            if data_1d is not None:
                # Reshape (using order='F' for R compliance) and plot
                data_2d = data_1d.reshape(som_shape, order='F')
                ax.imshow(
                    data_2d, 
                    cmap=cmap, 
                    origin='lower', 
                    vmin=vmin, 
                    vmax=vmax,
                    interpolation='nearest'
                )
            
            # Remove borders and ticks for a clean grid look
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

    # Add a global title
    #fig.suptitle('SOM Portraits by Cell Type and RNA Batch', fontsize=16, fontweight='bold', y=1.05)
    
    return fig
# ==========================================
# EXAMPLE USAGE
# ==========================================
# # Define your custom palettes (placeholders below)
# lymphoma_ontogeny_palette = {
#     'Bone_marrow_CD19+_': '#E6E6FA', 'Immature_': '#D8BFD8', 'Naive_': '#ADD8E6', 
#     'GC_': '#87CEEB', 'Centroblast_': '#4682B4', 'Centrocyte_': '#4169E1',
#     'Follicular_Lymphoma_': '#0000CD', 'MZ_': '#20B2AA', 'Memory_': '#3CB371',
#     'Plasmablast_': '#FFA07A', 'Plasma_': '#FF6347', 'B_cells_': '#A9A9A9'
# }
# 
# rna_batch_palette = {
#     'GPL570_FF_Unknown': '#FFD700', 'RNASeq_FF_rRNADepletion': '#FF8C00',
#     'GPL13158_FF_Unknown': '#DAA520', 'RNASeq_FF_PolyA': '#CD853F',
#     'RNASeq_FF_Unknown': '#8B4513', 'GPL17077_FF_Unknown': '#A0522D',
#     'GPL16686_FF_Unknown': '#D2691E', 'GPL23541_FF_Unknown': '#B8860B',
#     'GPL20188_FF_Unknown': '#CD5C5C', 'GPL96+97_FF_Unknown': '#F4A460',
#     'RNASeq_FF_Total': '#DEB887', 'GPL10739_FF_Unknown': '#D2B48C',
#     'GPL96_FF_Unknown': '#BC8F8F', 'GPL17047_FF_Unknown': '#F08080',
#     'GPL26356_FF_Unknown': '#FA8072', 'GPL6244_FF_Unknown': '#E9967A',
#     'RNASeq_FF_Exome_capture': '#8B0000'
# }
# 
# fig = plot_som_grid(
#     metagenes_df=metagenes_df, 
#     annotation_df=comb_ann_normal,
#     cell_type_col='Diagnosis_unified_with_coo',
#     batch_col='RNA_BATCH',
#     lymphoma_ontogeny_palette=lymphoma_ontogeny_palette,
#     rna_batch_palette=rna_batch_palette,
#     figsize_per_cell=(1.5, 1.5) # Adjust this size depending on monitor resolution
# )
# plt.show()

In [ ]:
fig = plot_som_grid(
    metagenes_df=metagenes_df, 
    annotation_df=comb_ann_normal,
    cell_type_col='Diagnosis_unified_with_coo',
    batch_col='RNA_BATCH',
    lymphoma_ontogeny_palette=lymphoma_ontogeny_palette,
    rna_batch_palette=rna_batch_palette,
    vmin=-10000,vmax=10000,
    figsize_per_cell=(0.5, 0.5) # Adjust this size depending on monitor resolution
 )
plt.show()

In [ ]:
fig = plot_som_grid(
    metagenes_df=metagenes_df, 
    annotation_df=comb_ann_dedup[~comb_ann_dedup.Major_group.isin(['Kassandra', 'Normal_B_cells'])],
    cell_type_col='Diagnosis_unified_with_coo',
    batch_col='RNA_BATCH',
    lymphoma_ontogeny_palette=lymphoma_ontogeny_palette,
    rna_batch_palette=rna_batch_palette,
    vmin=-10000,vmax=10000,
    figsize_per_cell=(0.5, 0.5) # Adjust this size depending on monitor resolution
 )
plt.show()

### Apply monocle